<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
# 9 · Reinforcement and Preference Learning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/09-preference-learning.ipynb)

**From Traditional NLP to Modern LLMs** · Day 3 · 120 minutes (55 briefing, 55 lab, 10 debrief)

Text generation as a reinforcement-learning problem, the policy gradient, and reward models learned from comparisons.

**By the end of this notebook you can:**

- Frame text generation as a reinforcement-learning problem
- Derive and implement the policy gradient
- Train a reward model from pairwise preferences

**Stack:** PyTorch, Hugging Face

In [ ]:
#@title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Tick WORKED_EXAMPLE to run every reference solution in place of your code. That gives a
# complete worked example; it does not show that you did the exercises.
WORKED_EXAMPLE = False  #@param {type:"boolean"}

import os as _os
import time as _time

from IPython import get_ipython as _get_ipython

_NOTEBOOK = "09-preference-learning"
_CONTENT_SHA = "5d88a5e70d08d562"
_EXERCISES = {1: ('returns_to_go', 'reinforce_loss'), 2: ('loo_baseline',), 3: ('bt_prob',), 4: ('bt_loss',), 5: ('fit_normalizer',)}
_HINTS = {1, 2, 4}  # exercises with a folded Hint cell above their TODO cell
_PATHS = {'offline': 'offline stand-ins and test doubles: checks that the code runs, not what a model does', 'open': 'real open models from the Hugging Face Hub, no API keys', 'keyed': 'commercial APIs with keys'}
_FALLBACK_NOTE = None
_RECORDED_SETTINGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE', 'NLP_LLMS_QUICK', 'HF_HUB_OFFLINE')
_OFFLINE_FLAGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE')
_FALLBACK_MARKERS = ('Could not load ', 'did not load', 'USING StubProvider', 'LSA stand-in fitted', 'OFFLINE TEST MODE')
_RECORD_PACKAGES = (
    "torch", "transformers", "numpy", "scikit-learn", "peft", "datasets", "openai",
    "anthropic", "typesafe-sdk", "langgraph", "langchain-core", "llama-index-core",
)


class _Workshop:
    """Keeps the folded solutions from replacing your code, and says whose code each
    checkpoint checked: yours, the reference solution, or code the lab provides."""

    _MISSING = object()

    def __init__(self, old=None):
        # Rerunning this cell (as Run all does) keeps the stored solutions but starts a new
        # run: the checkpoint results, cell times and run clock describe this run only.
        self.ref = getattr(old, "ref", {})
        self.stub = getattr(old, "stub", {})
        self.chosen = getattr(old, "chosen", set())  # exercises switched by use_reference
        self.verified = {}
        self._verifying = None
        self.results = {}
        self.cell_seconds = []
        self.cell_errors = 0
        self.fallbacks = set()
        self.started = _time.time()
        self._current = None
        self._t0 = None

    @property
    def worked(self):
        if _os.environ.get("NLP_LLMS_WORKED") == "1":
            return True
        return bool(_get_ipython().user_ns.get("WORKED_EXAMPLE"))

    def _is_reference(self, n, name, obj):
        return any(obj is r for r in self.ref.get(n, {}).get(name, ()))

    def solution(self, n):
        """Decorator on a reference solution: stores it and, unless WORKED_EXAMPLE is
        ticked, leaves your own definition of the same name in place."""

        def bind(obj, name=None):
            name = name or obj.__name__
            self.ref.setdefault(n, {}).setdefault(name, []).append(obj)
            current = _get_ipython().user_ns.get(name, self._MISSING)
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.stub.setdefault(n, {})[name] = current
            if self.worked:
                return obj
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.chosen.discard(n)  # your TODO cell ran again: back to your code
                return current
            if n in self.chosen:
                return obj  # you chose the reference with use_reference(n)
            # Your TODO cell has not run, or the name still holds a reference from a
            # worked run: go back to your code if it is known, never silently to the reference.
            yours = self.stub.get(n, {}).get(name, self._MISSING)
            if yours is not self._MISSING:
                return yours
            if callable(obj):
                return self._not_run(n, name)
            print(f"[workshop] Run your # TODO {n} cell: {name} is the reference until you do.")
            return obj

        return bind

    def _not_run(self, n, name):
        """Stands in for a function or class whose TODO cell has not run yet."""

        def missing(*args, **kwargs):
            raise NotImplementedError(
                f"{name}: run your # TODO {n} cell first, or go on with"
                f" workshop.use_reference({n!r})"
            )

        missing.__name__ = name
        return missing

    def solution_value(self, n, name, value):
        """The same as solution(n), for a value rather than a function or class."""
        if value is None or isinstance(value, (bool, int, float, str, bytes)):
            # Python shares equal small values, so yours and the reference could not be told
            # apart: give such a value in the stub instead of making it part of the exercise.
            kind = type(value).__name__
            raise TypeError(f"solution_value({n!r}, {name!r}): use an object, not a plain {kind}")
        return self.solution(n)(value, name)

    def use_reference(self, n):
        """Go on with the reference solution of exercise n in place of yours."""
        if n not in self.ref:
            raise KeyError(f"Exercise {n} has no reference solution yet: run its Solution cell")
        ns = _get_ipython().user_ns
        for name, objs in self.ref[n].items():
            ns[name] = objs[-1]
        self.chosen.add(n)
        print(
            f"Exercise {n}: now using the REFERENCE solution. Rerun the cells below it."
            f" To switch back to your code, rerun your # TODO {n} cell."
        )

    def checkpoint(self, n=None, label=None):
        """First line of a checkpoint cell: notes whose code the cell is about to check."""
        names = _EXERCISES.get(n, ())
        ns = _get_ipython().user_ns
        k = sum(self._is_reference(n, x, ns.get(x, self._MISSING)) for x in names)
        if not names:
            whose = None  # the results of the cells above: provided code, maybe your earlier work
        elif k == len(names):
            whose = "the REFERENCE solution"
        elif k == 0:
            whose = "your code"
        else:
            whose = "partly the REFERENCE solution"
        self._current = (n, str(label if label is not None else n), whose)

    def _pre(self, info):
        self._t0 = _time.time()

    def _post(self, result):
        if self._t0 is None:  # this harness cell itself: its hooks were registered mid-cell
            return
        self.cell_seconds.append(round(_time.time() - self._t0, 2))
        if not result.success and self._verifying is None:
            self.cell_errors += 1  # a verification copy fails on purpose; it is not an error
        if self._current is None:
            return
        n, label, whose = self._current
        self._current = None
        if self._verifying is not None:
            self.verified[self._verifying] = bool(result.success)
            return
        self.results[label] = (bool(result.success), whose)
        if whose is None:
            if result.success:
                print(f"[workshop] Checkpoint {label} passed.")
            elif isinstance(result.error_in_exec, NotImplementedError):
                print(
                    f"[workshop] Checkpoint {label} uses an exercise you have not written yet."
                    " Finish it, or go on with workshop.use_reference(N) for that exercise."
                )
            else:
                print(f"[workshop] Checkpoint {label} failed. Read the message above.")
        elif result.success:
            print(f"[workshop] Checkpoint {label} passed on {whose}.")
        elif isinstance(result.error_in_exec, NotImplementedError):
            print(
                f"[workshop] Checkpoint {label}: TODO {n} is not written yet. Write it, run"
                " its cell, then rerun this one. To go on without it:"
                f" workshop.use_reference({n!r})"
            )
            if n in _HINTS:
                print(f"[workshop] Stuck? Open the folded Hint above TODO {n} first.")
        elif whose == "the REFERENCE solution":
            print(
                f"[workshop] Checkpoint {label} failed on the reference solution: a problem"
                " with the lab or the runtime, not with your code. Tell the instructor."
            )
        else:
            print(
                f"[workshop] Checkpoint {label} failed on {whose}. Read the assertion above,"
                f" fix TODO {n}, or go on with workshop.use_reference({n!r})."
            )
            if n in _HINTS:
                print(f"[workshop] The folded Hint above TODO {n} says what to check.")

    def _verify_begin(self, n):
        """Test-only (scripts/test_notebooks.py --verify-checkpoints): bind exercise n's
        stubs, so that a copy of its checkpoint runs on unfinished code."""
        ns = _get_ipython().user_ns
        self._saved = {name: ns.get(name, self._MISSING) for name in _EXERCISES.get(n, ())}
        ns.update(self.stub.get(n, {}))
        self._verifying = n

    def _verify_end(self, n):
        """Test-only: restore the reference and require that the stub copy failed."""
        ns = _get_ipython().user_ns
        for name, value in self._saved.items():
            if value is self._MISSING:
                ns.pop(name, None)
            else:
                ns[name] = value
        self._verifying = None
        if self.verified.get(n) is not False:
            raise AssertionError(
                f"The checkpoint of exercise {n} passed on the unfinished stub: it cannot tell"
                " a finished exercise from an unfinished one"
            )

    def summary(self):
        """Which checkpoints passed, and on whose code."""
        groups = {"your code": [], "the reference": [], "other checks": [], "failed": []}
        for label, (ok, whose) in self.results.items():
            if not ok:
                groups["failed"].append(label)
            elif whose is None:
                groups["other checks"].append(label)
            elif whose == "your code":
                groups["your code"].append(label)
            else:
                groups["the reference"].append(label)
        mode = "WORKED EXAMPLE: reference solutions" if self.worked else "your code"
        print(f"{_NOTEBOOK}, run with {mode}.")
        path = self.path()
        # _PATHS comes from readiness.paths in _variables.yml; the offline entry says that
        # such numbers check the code, not a model.
        print(f"  What ran: {_PATHS[path]}.")
        if path == "open" and _FALLBACK_NOTE:
            print(f"  Read the numbers with this in mind: {_FALLBACK_NOTE}.")
        lines = {
            "your code": "Checkpoints passed on your code",
            "the reference": "Checkpoints passed on a reference solution",
            "other checks": "Other checks passed",
            "failed": "Checkpoints failed",
        }
        for group, labels in groups.items():
            print(f"  {lines[group]}: {', '.join(labels) or 'none'}")
        if self.worked:
            print("  A worked-example run shows how the lab goes, not that you did it.")

    def path(self):
        """offline, keyed or open: the same rule as scripts/add_run_record.py's path_of."""
        ns = _get_ipython().user_ns
        if self.fallbacks or any(_os.environ.get(flag) for flag in _OFFLINE_FLAGS):
            return "offline"
        if ns.get("PROVIDER") in ("openai", "anthropic") or ns.get("JEV_PATH") == "keyed":
            return "keyed"
        return "open"

    def run_record(self):
        """A run record for runs/ (see runs/README.md): paste it into
        scripts/add_run_record.py. It holds no code, outputs, keys or names."""
        import importlib.metadata as _md
        import json
        import platform
        import sys

        gpu = None
        try:
            import torch

            if torch.cuda.is_available():
                gpu = torch.cuda.get_device_name(0)
        except Exception:
            pass
        packages = {}
        for name in _RECORD_PACKAGES:
            try:
                packages[name] = _md.version(name)
            except Exception:
                pass
        import datetime as _dt

        record = {
            "date": _dt.datetime.now(_dt.timezone.utc).date().isoformat(),
            "notebook": _NOTEBOOK,
            "content_sha": _CONTENT_SHA,
            "mode": "worked" if self.worked else "learner",
            # A run passes when it reached this last cell with no error on the way and every
            # checkpoint passed. Run all reruns the harness first, so it starts clean.
            "status": "pass"
            if self.results and self.cell_errors == 0 and all(ok for ok, _ in self.results.values())
            else "fail",
            "cell_errors": self.cell_errors,
            "fallbacks": sorted(self.fallbacks),
            "seconds": round(_time.time() - self.started, 1),
            "cell_seconds": self.cell_seconds,
            "python": platform.python_version(),
            "platform": sys.platform,
            "gpu": gpu,
            "colab_release": _os.environ.get("COLAB_RELEASE_TAG"),
            "provider": _get_ipython().user_ns.get("PROVIDER"),
            "jev_path": _get_ipython().user_ns.get("JEV_PATH"),
            "settings": {k: _os.environ[k] for k in _RECORDED_SETTINGS if _os.environ.get(k)},
            "packages": packages,
        }
        print("----- run record (paste into scripts/add_run_record.py) -----")
        print(json.dumps(record))
        print("----- end of run record -----")


_ip = _get_ipython()
_old = _ip.user_ns.get("workshop")
# Remove only an earlier harness's own hooks; every other hook (Colab's, an extension's)
# stays registered.
for _event in ("pre_run_cell", "post_run_cell"):
    for _callback in list(_ip.events.callbacks[_event]):
        if type(getattr(_callback, "__self__", None)).__name__ == "_Workshop":
            _ip.events.unregister(_event, _callback)
workshop = _Workshop(_old if hasattr(_old, "ref") else None)

# Notice when a lab falls back to a stand-in, for the run record. print is wrapped once.
import builtins as _builtins

if not getattr(_builtins.print, "_workshop", False):
    _print = _builtins.print

    def _watching_print(*args, **kwargs):
        try:
            ip = _get_ipython()
            ws = ip.user_ns.get("workshop") if ip is not None else None
            if ws is not None and hasattr(ws, "fallbacks"):
                # Labs print their fallback notices as strings; never stringify other objects.
                text = " ".join(a for a in args if isinstance(a, str))
                ws.fallbacks.update(m.strip() for m in _FALLBACK_MARKERS if m in text)
        except Exception:
            pass  # watching must never break printing
        return _print(*args, **kwargs)

    _watching_print._workshop = True
    _builtins.print = _watching_print
_ip.events.register("pre_run_cell", workshop._pre)
_ip.events.register("post_run_cell", workshop._post)
print("Workshop harness ready:", "WORKED EXAMPLE" if workshop.worked else "checking your code")


## Setup

This lab runs on a **CPU runtime**; a GPU makes the reward-model training of Exercise 4 a little faster but is not needed. It needs no API key, no installs and, in the core path, no model download: PyTorch, NumPy and matplotlib are preinstalled on Colab.

**Two halves.**

- **Part A** (Exercises 1 and 2): REINFORCE on a toy sequence task whose optimal policy and exact gradient are known, so every estimate can be checked against the truth.
- **Part B** (Exercises 3 to 5): a reward model trained from pairwise preferences. The pairs are continuations sampled from the small GPT-2 of Lab 6, the model Lab 10 will fine-tune, and labeled by a synthetic rater whose rule we wrote. You check that the reward model recovers that rule, then save it for Lab 10.

**How the lab works.** There are five exercises and one optional stretch section. Each exercise has:

1. a **Predict** question to answer before you run anything;
2. a `# TODO` cell where you write one or two short functions;
3. a folded **Solution** cell (open it only after trying);
4. a **Checkpoint** cell that tells you whether your function is right.

The folded solution cells never replace your code: each checkpoint checks your own functions and says so. If you are stuck, `workshop.use_reference(N)` lets you go on with exercise N's reference solution, and the checkpoint then says it checked the reference. To run the whole lab as a worked example, tick `WORKED_EXAMPLE` in the harness cell at the top; a worked run shows how the lab goes, not that you did it.

**What you write and what is provided.** You write six short functions: `returns_to_go`, `reinforce_loss`, `loo_baseline`, `bt_prob`, `bt_loss` and `fit_normalizer`. The toy environment, its exact gradient, the data loading, the gold rule, the reward model, the training loops, the evaluation and the plots are provided.

> **Instructor note (status of this notebook).** Part A was executed and its thresholds were set from five seeds on a CPU. Part B was run on the committed preference files (built 2026-10-06 by `data/build_lab09_preferences.py`, which needs the Hugging Face Hub), and its Exercise 4 thresholds were set from three seeds. Both were measured on an Apple M1 Pro CPU, not yet on Colab.

In [ ]:
# Setup: imports, seeds, device. Everything here is preinstalled on Colab.
import copy
import gzip
import hashlib
import itertools
import json
import math
import os
import random
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # used only by the reward model

# TEST-ONLY SWITCH. Participants: leave this alone; it is off unless the environment sets it.
# The real preference files are committed; the repository's offline CI leg still replaces them by a
# stand-in. With NLP_LLMS_OFFLINE_TINY=1 (or, for this lab only,
# NLP_LLMS_LAB09_OFFLINE=tiny), Part B runs on a small synthetic stand-in (random token ids from a
# local vocabulary, see Part B) so that every cell's code can be executed. Numbers printed in that
# mode are not results and must never be reported.
OFFLINE_TINY = (os.environ.get("NLP_LLMS_OFFLINE_TINY") == "1"
                or os.environ.get("NLP_LLMS_LAB09_OFFLINE") == "tiny")

NAVY, ACCENT, GRAY = "#16324f", "#b3541e", "#8a8a8a"  # plot colors
LAB_START = time.time()
TIMES = {}  # section -> seconds, printed at the end of the core path

print(f"PyTorch {torch.__version__} · device {DEVICE} · {torch.get_num_threads()} CPU threads"
      + (" · OFFLINE STAND-IN MODE (test only)" if OFFLINE_TINY else ""))

# Part A · The Policy Gradient on a Toy Task

## Exercise 0 · Read the Toy Environment (3 minutes)

Text generation is a sequence of decisions (briefing section 2): the state is everything written so far, the action is the next token, and the reward usually arrives at the end. Part A uses an environment small enough that we know the right answer.

| | Toy task | Text generation |
|---|---|---|
| vocabulary | 8 symbols, `0` to `7`, plus a start symbol `BOS` (index 8) | the tokenizer's vocabulary $V$ |
| episode | $T = 6$ actions | a response of $T$ tokens |
| state $s_t$ | the position $t$ and the previous symbol | the prompt and the tokens so far |
| policy $\pi_\theta$ | a table of logits `theta` of shape `(T, 9, 8)`: one row of 8 logits per (position, previous symbol). A "bigram policy", as in Module 1 | the language model |
| reward | the share of positions that match a hidden target `y_star`, given once at the end | a score of the whole response |

`theta` starts at zero, so training starts at the **uniform policy**. The optimal policy writes `y_star` with probability 1 and earns $J = 1$.

The provided functions, with the briefing's equations:

- `sample_toy(theta, N, generator)` draws $N$ episodes and returns the logits used at each step, `(N, T, 8)`, and the actions, `(N, T)`.
- `token_logprobs(logits, actions)` returns $\log \pi_\theta(y_t \mid s_t)$ for every step, `(N, T)`; summed over $t$ it is briefing eq. `seq-logprob`.
- `toy_rewards(actions, level)` returns `rewards` of shape `(N, T)`. With `level="sequence"` they are zero except the last column, $r_T = R = \frac{1}{T}\sum_t \mathbf{1}[y_t = y^\star_t]$; with `level="token"`, $r_t = \mathbf{1}[y_t = y^\star_t] / T$. Both have the same total.
- `exact_gradient(theta)` enumerates all $8^6 = 262{,}144$ responses, computes $J(\theta)$ exactly (briefing eq. `objective`) and backpropagates: the true $\nabla_\theta J$. This is only possible because the task is tiny.
- `expected_return(theta)` computes the same $J$ much faster, by propagating the distribution of the previous symbol through the six positions. Training uses it to monitor progress.
- `batch_mean_baseline(returns)`: the per-position mean over the batch, shape `(1, T)` (briefing section 4). Lab 10 uses it.

**Predict.** What is $J$ for the uniform policy? (Each position matches the target with what probability?)

In [ ]:
# Provided: the toy environment of Part A. Everything runs on the CPU.
N_SYMBOLS, T, BOS = 8, 6, 8  # 8 actions, episodes of T = 6, the start symbol is index 8
y_star = torch.randint(0, N_SYMBOLS, (T,), generator=torch.Generator().manual_seed(0))  # target


def sample_toy(theta, N, generator):
    """Draw N episodes from the bigram policy theta (T, 9, 8).

    Returns logits (N, T, 8), differentiable with respect to theta, and actions (N, T).
    """
    prev = torch.full((N,), BOS, dtype=torch.long)  # every episode starts after BOS
    logits, actions = [], []
    for t in range(T):
        z = theta[t, prev]  # (N, 8): the row for (position t, previous symbol)
        a = torch.multinomial(torch.softmax(z.detach(), dim=-1), 1, generator=generator)[:, 0]
        logits.append(z)
        actions.append(a)
        prev = a
    return torch.stack(logits, dim=1), torch.stack(actions, dim=1)


def token_logprobs(logits, actions):
    """log pi(y_t | s_t) for every step: logits (N, T, 8), actions (N, T) -> (N, T)."""
    return F.log_softmax(logits, dim=-1).gather(-1, actions[..., None])[..., 0]


def toy_rewards(actions, level="sequence"):
    """rewards (N, T). "sequence": all zero but r_T = share of matches; "token": match / T."""
    match = (actions == y_star).float()  # (N, T)
    if level == "token":
        return match / T
    rewards = torch.zeros_like(match)
    rewards[:, -1] = match.mean(dim=-1)  # R(y), given once at the end
    return rewards


ALL_RESPONSES = torch.tensor(list(itertools.product(range(N_SYMBOLS), repeat=T)))  # (8^6, T)
_ALL_PREV = torch.cat([torch.full((len(ALL_RESPONSES), 1), BOS), ALL_RESPONSES[:, :-1]], dim=1)
_ALL_R = (ALL_RESPONSES == y_star).float().mean(dim=-1)  # R(y) for every response


def exact_objective(theta):
    """J(theta) = sum_y pi(y) R(y) over all 8^6 responses (briefing eq. objective)."""
    log_pi = F.log_softmax(theta, dim=-1)[torch.arange(T), _ALL_PREV, ALL_RESPONSES].sum(-1)
    return (log_pi.exp() * _ALL_R).sum()


def exact_gradient(theta):
    """The true gradient of J at theta, by enumeration and autograd. Shape (T, 9, 8)."""
    theta = theta.detach().clone().requires_grad_(True)
    exact_objective(theta).backward()
    return theta.grad


@torch.no_grad()
def expected_return(theta):
    """J(theta) exactly, in O(T * 9 * 8): propagate the distribution of the previous symbol."""
    probs = torch.softmax(theta, dim=-1)  # (T, 9, 8)
    prev = torch.zeros(N_SYMBOLS + 1)
    prev[BOS] = 1.0
    J = 0.0
    for t in range(T):
        marginal = prev @ probs[t]  # P(y_t = a) for a = 0..7
        J += marginal[y_star[t]].item() / T
        prev = torch.cat([marginal, torch.zeros(1)])  # y_t is the next step's previous symbol
    return J


def batch_mean_baseline(returns):
    """(N, T) -> (1, T): per position, the mean over the batch (briefing section 4)."""
    return returns.mean(dim=0, keepdim=True)


theta_uniform = torch.zeros(T, N_SYMBOLS + 1, N_SYMBOLS)  # all logits 0: the uniform policy
started = time.time()
g_exact = exact_gradient(theta_uniform)
exact_seconds = time.time() - started
print("y_star =", y_star.tolist())
print(f"J(uniform): by enumeration {exact_objective(theta_uniform).item():.4f}, "
      f"by expected_return {expected_return(theta_uniform):.4f}")
print(f"exact_gradient over {len(ALL_RESPONSES):,} responses: {exact_seconds:.2f} s")
assert y_star.tolist() == [4, 7, 5, 0, 3, 3], "the target is fixed by the seed"
assert abs(expected_return(theta_uniform) - 1 / N_SYMBOLS) < 1e-6

**Explain.** Under the uniform policy each position matches its target with probability $1/8$, and the reward is the mean of six such matches, so $J = 1/8 = 0.125$ whatever `y_star` is. The two ways of computing $J$ agree. `exact_gradient` is affordable here (a fraction of a second) only because there are $8^6$ responses; a 24-token response from GPT-2 has $50{,}256^{24}$, which is why the rest of the module estimates the gradient by sampling.

## Exercise 1 · Returns and the REINFORCE Loss (12 minutes)

The return from step $t$ is the reward collected from then on (briefing eq. `return`):

$$
G_t = \sum_{t'=t}^{T} r_{t'} .
$$

The REINFORCE loss is the negative log-likelihood of the model's own samples, each token weighted by its return minus a baseline (briefing eqs. `pg-loss` and `reward-to-go`), averaged over the $N$ samples:

$$
\mathcal{L}_{\text{PG}}(\theta) = -\frac{1}{N} \sum_{i=1}^{N} \sum_{t=1}^{T} \big(G^{(i)}_t - b^{(i)}_t\big)\, \log \pi_\theta\big(y^{(i)}_t \mid s^{(i)}_t\big).
$$

The weights $G_t - b_t$ are constants: the reward is never differentiated. Its gradient is $-\hat g$, the REINFORCE estimate (briefing eq. `reinforce`).

**Predict.** For rewards `[[1, 0, 2]]`, what are the returns? With a sequence-level reward (zero except the last column, $R$), what is $G_t$ at every step?

**Task.** Complete both functions.

- `returns_to_go(rewards)`: `(N, T) -> (N, T)`. One line: flip along time, cumulative sum, flip back.
- `reinforce_loss(logp, returns, baseline=0.0)`: `logp` and `returns` of shape `(N, T)`; `baseline` is a scalar or a tensor that broadcasts to `(N, T)`. Detach the weights, sum over $t$, average over the samples.

<details>
<summary>Hint for TODO 1</summary>

$G_t$ adds the rewards from $t$ to the end (briefing eq. `return`), so it is a running sum taken backwards in time: a plain cumulative sum gives $r_1 + \dots + r_t$, which turns `[[1, 0, 2]]` into `[[1, 1, 3]]` instead of `[[3, 2, 2]]`. In the loss, detach `returns - baseline` so that the reward is never differentiated (eq. `pg-loss`), sum over $t$, and average over the $N$ samples.

</details>

In [ ]:
# TODO 1: complete both functions.
def returns_to_go(rewards):
    """G_t = r_t + r_{t+1} + ... + r_T. rewards: (N, T) -> (N, T)."""
    raise NotImplementedError("TODO 1")


def reinforce_loss(logp, returns, baseline=0.0):
    """-((returns - baseline) * logp).sum(-1).mean(), with the weights treated as constants.

    logp, returns: (N, T). baseline: scalar, (1, T) or (N, T).
    """
    raise NotImplementedError("TODO 1")

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
def returns_to_go(rewards):
    """G_t = r_t + r_{t+1} + ... + r_T. rewards: (N, T) -> (N, T)."""
    return rewards.flip(-1).cumsum(-1).flip(-1)  # reversed running sum along time


@workshop.solution(1)
def reinforce_loss(logp, returns, baseline=0.0):
    """-((returns - baseline) * logp).sum(-1).mean(), with the weights treated as constants.

    logp, returns: (N, T). baseline: scalar, (1, T) or (N, T).
    """
    weight = (returns - baseline).detach()  # constants: R is never differentiated
    return -(weight * logp).sum(-1).mean()  # sum over t, mean over samples

<details>
<summary>Why this solution works</summary>

Flipping the time axis turns "sum from $t$ to the end" into "sum from the start to $t$", which is `cumsum`; flipping back puts each $G_t$ at position $t$. For a sequence-level reward only the last column is nonzero, so every $G_t$ equals $R$: every token gets the same credit (briefing section 5).

`.detach()` makes the weights constants, so autograd differentiates only `logp`. The gradient of the loss is then $-\frac{1}{N}\sum_i \sum_t (G^{(i)}_t - b^{(i)}_t)\, \nabla_\theta \log \pi_\theta(y^{(i)}_t \mid s^{(i)}_t)$, minus the REINFORCE estimate. Without the detach, a baseline computed from the returns would not change anything here (returns do not depend on $\theta$ in this task), but in Lab 10 the returns include a KL term computed from the policy, and differentiating it would add a wrong term to the gradient.

</details>

In [ ]:
# Checkpoint 1a: hand cases.
workshop.checkpoint(1, label="1a")
assert torch.equal(returns_to_go(torch.tensor([[1.0, 0.0, 2.0]])), torch.tensor([[3.0, 2.0, 2.0]])), (
    f"the returns of rewards [[1, 0, 2]] are [[3, 2, 2]], got {returns_to_go(torch.tensor([[1.0, 0.0, 2.0]])).tolist()}: "
    "G_t sums the rewards from t to the end")
_seq = torch.tensor([[0.0, 0.0, 0.5], [0.0, 0.0, 1.0]])  # sequence-level rewards
assert torch.equal(returns_to_go(_seq), torch.tensor([[0.5] * 3, [1.0] * 3])), "G_t = R at every t"

_logp = torch.tensor([[-1.0, -2.0, -0.5], [-0.1, -0.2, -0.3]], requires_grad=True)  # (2, 3)
_returns = torch.tensor([[3.0, 2.0, 2.0], [1.0, 1.0, 1.0]], requires_grad=True)
_loss = reinforce_loss(_logp, _returns)
# by hand: sample 1: -(3*-1 + 2*-2 + 2*-0.5) = 8; sample 2: -(-0.1 - 0.2 - 0.3) = 0.6; mean 4.3
assert abs(_loss.item() - 4.3) < 1e-6, (
    f"by hand the loss is the mean of 8 and 0.6, 4.3; got {_loss.item()}: sum over t, average over the samples, "
    "and keep the minus sign")
_baseline = torch.tensor([[1.0, 1.0, 1.0]], requires_grad=True)
_loss_b = reinforce_loss(_logp, _returns, _baseline)
assert abs(_loss_b.item() - (-(2 * -1 + 1 * -2 + 1 * -0.5) + 0.0) / 2) < 1e-6, (
    f"with baseline 1 the loss is (4.5 + 0) / 2 = 2.25, got {_loss_b.item()}: weight each log-probability by "
    "returns - baseline")
_loss_b.backward()
assert _returns.grad is None and _baseline.grad is None, "detach the weights: no gradient to returns"
assert torch.allclose(_logp.grad, -torch.tensor([[2.0, 1.0, 1.0], [0.0, 0.0, 0.0]]) / 2), (
    f"the gradient with respect to logp must be -(returns - baseline) / N = [[-1, -0.5, -0.5], [0, 0, 0]], "
    f"got {_logp.grad.tolist()}")
print("Checkpoint 1a passed")

**Run: is the estimate right on average?** REINFORCE is unbiased: the mean of many estimates is the true gradient (briefing eq. `reinforce`). The provided `estimate_gradients(theta, K, N, kind)` draws $K$ independent batches of $N$ episodes and returns one gradient estimate $\hat g$ per batch, computed with *your* two functions: `kind="none"` means no baseline and a sequence-level reward. We compare the mean of $K = 500$ estimates (batches of $N = 16$) with `exact_gradient` at the uniform policy, by cosine similarity.

**Predict.** Will a single batch of 16 point in roughly the right direction? Will the mean of 500 batches?

In [ ]:
# Provided: K independent gradient estimates, each from one batch of N episodes.
def gradient_estimate(theta, N, generator, kind="none"):
    """One REINFORCE estimate g_hat of grad J (the negative gradient of your loss), (T, 9, 8).

    kind: "none" (no baseline), "loo" (your leave-one-out baseline), "batch_mean", or
    "token_loo" (token-level rewards through your returns_to_go, with leave-one-out).
    """
    theta = theta.detach().clone().requires_grad_(True)
    logits, actions = sample_toy(theta, N, generator)
    logp = token_logprobs(logits, actions)  # (N, T)
    returns = returns_to_go(toy_rewards(actions, "token" if kind == "token_loo" else "sequence"))
    baseline = {"none": lambda returns: 0.0,  # loo_baseline is looked up only when used
                "loo": lambda returns: loo_baseline(returns),
                "token_loo": lambda returns: loo_baseline(returns),
                "batch_mean": batch_mean_baseline}[kind]
    (grad,) = torch.autograd.grad(reinforce_loss(logp, returns, baseline(returns)), theta)
    return -grad  # the loss is minimized, J is maximized


def estimate_gradients(theta, K, N, kind="none", seed=SEED):
    """(K, T * 9 * 8): one flattened estimate per batch, K independent batches."""
    generator = torch.Generator().manual_seed(seed)
    return torch.stack([gradient_estimate(theta, N, generator, kind).reshape(-1) for _ in range(K)])


def cosine(a, b):
    return (a @ b / (a.norm() * b.norm())).item()


K, N = 500, 16
started = time.time()
estimates = {"none": estimate_gradients(theta_uniform, K, N, "none")}
g_flat = g_exact.reshape(-1)
mean_none = estimates["none"].mean(dim=0)
single = [cosine(estimates["none"][k], g_flat) for k in range(K)]
print(f"{K} estimates of batch size {N} in {time.time() - started:.1f} s")
print(f"cosine(one batch, exact):  median {np.median(single):.3f}, range {min(single):.3f} to {max(single):.3f}")
print(f"cosine(mean of {K}, exact): {cosine(mean_none, g_flat):.4f}; "
      f"relative error {((mean_none - g_flat).norm() / g_flat.norm()).item():.3f}")

In [ ]:
# Checkpoint 1b
workshop.checkpoint(1, label="1b")
COSINE_FLOOR = 0.95  # measured 0.983 to 0.984 at seeds 0 to 4 (see the notes after Exercise 2)
assert cosine(mean_none, g_flat) >= COSINE_FLOOR, "the mean estimate should match the exact gradient"
print("Checkpoint 1b passed: REINFORCE is right on average")

**Explain.** One batch of 16 is a poor estimate: its cosine with the true gradient is far below 1, and it varies a lot from batch to batch. The mean of 500 batches points almost exactly along the true gradient. Unbiased does not mean accurate: it means right *on average*. Training takes one noisy step per batch, and the noise is what Exercise 2 attacks.

**Run: train the policy.** The provided `train_toy` repeats: sample $N = 16$ episodes, compute your loss, take one Adam step (learning rate 0.05), for 300 steps. It records the exact $J$ after every step with `expected_return`. Sequence-level reward, no baseline.

**Predict.** Will $J$ reach the optimum of 1 in 300 steps? Roughly how many steps to reach 0.9?

In [ ]:
# Provided: the REINFORCE training loop of Part A.
def train_toy(baseline_fn=None, seed=SEED, steps=300, lr=0.05, N=16, level="sequence"):
    """Train theta from the uniform policy. Returns (theta, J after each step, steps to J >= 0.9)."""
    generator = torch.Generator().manual_seed(seed)
    theta = torch.zeros(T, N_SYMBOLS + 1, N_SYMBOLS, requires_grad=True)
    optimizer = torch.optim.Adam([theta], lr=lr)
    history = [expected_return(theta)]
    for _ in range(steps):
        logits, actions = sample_toy(theta, N, generator)
        logp = token_logprobs(logits, actions)  # (N, T)
        returns = returns_to_go(toy_rewards(actions, level))  # (N, T)
        baseline = baseline_fn(returns) if baseline_fn is not None else 0.0
        loss = reinforce_loss(logp, returns, baseline)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(expected_return(theta))
    first = next((step for step, J in enumerate(history) if J >= 0.9), None)
    return theta.detach(), history, first


started = time.time()
theta_trained, history_none, first_none = train_toy(seed=SEED)
print(f"300 steps in {time.time() - started:.1f} s · final J = {history_none[-1]:.3f} · "
      f"first J >= 0.9 at step {first_none}")
probs = torch.softmax(theta_trained, dim=-1)
greedy, prev = [], BOS
for t in range(T):
    greedy.append(int(probs[t, prev].argmax()))
    prev = greedy[-1]
print("most likely response:", greedy, "· target:", y_star.tolist())

plt.figure(figsize=(6, 3.2))
plt.plot(history_none, color=GRAY, label="no baseline")
plt.axhline(1.0, color="black", lw=0.8, ls=":")
plt.axhline(1 / 8, color="black", lw=0.8, ls=":")
plt.xlabel("step")
plt.ylabel("exact J")
plt.title("REINFORCE on the toy task")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Checkpoint 1c
workshop.checkpoint(1, label="1c")
J_FLOOR = 0.85  # final J was 0.91 to 0.98 at seeds 0 to 4, and at least 0.895 at seeds 0 to 19
assert history_none[-1] >= J_FLOOR, history_none[-1]
print(f"Checkpoint 1c passed: J rose from 0.125 to {history_none[-1]:.3f}")
TIMES["setup and Exercises 0-1"] = time.time() - LAB_START

**Explain.** $J$ climbs from $0.125$ toward 1, and the most likely response becomes the target. No gradient of the reward was ever computed: the reward entered only as a number weighting the log-probabilities of the policy's own samples. The curve is noisy and slow at first: early on few samples match many positions, and with a sequence-level reward every token of a sample gets the same credit.

## Exercise 2 · A Baseline, and the Variance It Removes (9 minutes)

The score function has mean zero (briefing eq. `score-zero`), so subtracting any baseline $b$ that does not depend on the sample it is subtracted from leaves the expected gradient unchanged (eq. `baseline`). The **leave-one-out** baseline uses the mean return of the *other* samples (eq. `loo`):

$$
b^{(i)}_t = \frac{1}{N - 1} \sum_{j \neq i} G^{(j)}_t .
$$

The batch mean, $\bar G_t = \frac{1}{N} \sum_j G^{(j)}_t$, includes the sample's own return. The briefing showed $G^{(i)}_t - \bar G_t = (1 - \frac{1}{N})\,(G^{(i)}_t - b^{(i)}_t)$: the batch-mean estimate is the leave-one-out estimate times $1 - 1/N$, on every batch.

**Predict.** The cell after the checkpoint measures the **total variance** of $\hat g$ (the sum over all 432 entries of `theta` of the variance across the 500 batches) at the uniform policy for three estimators:

1. no baseline, sequence-level reward;
2. your leave-one-out baseline, sequence-level reward;
3. your leave-one-out baseline with token-level rewards, through `returns_to_go` (each token credited only with the matches from its own position on).

Rank them from most to least variance. Roughly what fraction of the variance does the baseline remove?

**Task.** Complete `loo_baseline(returns)`: `(N, T) -> (N, T)`, with no loop. The sum over all samples minus the sample's own return is the sum over the others.

<details>
<summary>Hint for TODO 2</summary>

The other samples' sum is the per-position total over the batch minus the sample's own return; divide it by $N - 1$, the number of other samples, not by $N$ (briefing eq. `loo`). In the checkpoint's batch `[[1, 2], [3, 4], [5, 9]]`, the first sample's baseline is $((3 + 5)/2,\ (4 + 9)/2) = (4, 6.5)$. Keeping the sample's own return gives the batch mean, which depends on that sample.

</details>

In [ ]:
# TODO 2: complete this function.
def loo_baseline(returns):
    """b_t^(i) = mean of returns[j, t] over the other samples j != i. (N, T) -> (N, T); N >= 2."""
    raise NotImplementedError("TODO 2")

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def loo_baseline(returns):
    """b_t^(i) = mean of returns[j, t] over the other samples j != i. (N, T) -> (N, T); N >= 2."""
    N = returns.shape[0]
    return (returns.sum(0, keepdim=True) - returns) / (N - 1)  # (1, T) - (N, T): the others' sum

<details>
<summary>Why this solution works</summary>

`returns.sum(0, keepdim=True)` is the per-position total over the batch, shape `(1, T)`. Subtracting `returns` (broadcast to `(N, T)`) removes each sample's own return, leaving the sum over the other $N - 1$ samples; dividing gives their mean. Because the samples are drawn independently, $b^{(i)}$ is a function of the *other* samples only, so it does not depend on $y^{(i)}$ and eq. `baseline` applies: the estimate stays exactly unbiased.

</details>

In [ ]:
# Checkpoint 2a and 2b
workshop.checkpoint(2, label="2a and 2b")
_returns = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 9.0]])
assert torch.allclose(loo_baseline(_returns), torch.tensor([[4.0, 6.5], [3.0, 5.5], [2.0, 3.0]])), (
    f"the leave-one-out baselines of [[1, 2], [3, 4], [5, 9]] are [[4, 6.5], [3, 5.5], [2, 3]], "
    f"got {loo_baseline(_returns).tolist()}: subtract the sample's own return and divide by N - 1")

# 2b: an exact identity on one batch, no sampling noise: batch mean = (1 - 1/N) * leave-one-out.
for _seed in range(3):
    _g_loo = gradient_estimate(theta_uniform, N, torch.Generator().manual_seed(_seed), "loo")
    _g_mean = gradient_estimate(theta_uniform, N, torch.Generator().manual_seed(_seed), "batch_mean")
    assert torch.allclose(_g_mean, (1 - 1 / N) * _g_loo, atol=1e-6), "the identity of briefing section 4"
print(f"Checkpoint 2a and 2b passed: on the same batch, batch-mean gradient = (1 - 1/{N}) x leave-one-out")

# 2c: unbiased. The mean of K estimates with your baseline matches the exact gradient.
started = time.time()
for kind in ("loo", "token_loo", "batch_mean"):
    estimates[kind] = estimate_gradients(theta_uniform, K, N, kind)
variance_seconds = time.time() - started
for kind in ("none", "loo", "token_loo"):
    assert cosine(estimates[kind].mean(0), g_flat) >= COSINE_FLOOR, kind
print("Checkpoint 2c passed: every estimator is right on average "
      + ", ".join(f"({k}: cosine {cosine(estimates[k].mean(0), g_flat):.4f})" for k in ("loo", "token_loo")))

In [ ]:
# Run: the variance table, on the same K = 500 batches per estimator.
total_variance = {kind: estimates[kind].var(dim=0).sum().item() for kind in estimates}
rows = [("no baseline, sequence-level reward", "none"),
        ("leave-one-out baseline (yours), sequence-level", "loo"),
        ("leave-one-out, token-level rewards-to-go", "token_loo"),
        ("batch-mean baseline, sequence-level (Lab 10)", "batch_mean")]
print(f"Total variance of g_hat at the uniform policy, K = {K} batches of N = {N} ({variance_seconds:.1f} s)")
print(f"{'estimator':<50} {'total var':>10} {'ratio':>7} {'cosine(mean, exact)':>20}")
for name, kind in rows:
    print(f"{name:<50} {total_variance[kind]:>10.5f} {total_variance[kind] / total_variance['none']:>7.2f}"
          f" {cosine(estimates[kind].mean(0), g_flat):>20.4f}")
print(f"(1 - 1/N)^2 = {(1 - 1 / N) ** 2:.4f}: the batch-mean variance is the leave-one-out variance times this")

# One coordinate for the briefing's figure: theta[0, BOS, y_star[0]], at the uniform policy.
COORD = (0, BOS, int(y_star[0]))
flat_index = int(np.ravel_multi_index(COORD, tuple(theta_uniform.shape)))
coordinate = {kind: estimates[kind][:, flat_index] for kind in ("none", "loo")}
exact_value = g_exact[COORD].item()
figure_data = {
    "source": "notebooks/09-preference-learning.ipynb, Exercise 2 (generated by the notebook)",
    "description": ("K one-batch REINFORCE estimates of one coordinate of grad J at the uniform "
                    "policy, without a baseline and with the leave-one-out baseline"),
    "coordinate": {"t": COORD[0], "previous_symbol": "BOS", "action": COORD[2]},
    "K": K, "N": N, "seed": SEED, "reward": "sequence-level", "torch": torch.__version__,
    "exact": round(exact_value, 6),
    "no_baseline": [round(v, 6) for v in coordinate["none"].tolist()],
    "leave_one_out": [round(v, 6) for v in coordinate["loo"].tolist()],
    "std": {"no_baseline": round(coordinate["none"].std().item(), 6),
            "leave_one_out": round(coordinate["loo"].std().item(), 6)},
    "total_variance": {kind: round(v, 6) for kind, v in total_variance.items()},
}
with open("09-baseline-variance.json", "w", encoding="utf-8") as f:
    json.dump(figure_data, f, indent=1)

bins = np.linspace(min(v.min().item() for v in coordinate.values()),
                   max(v.max().item() for v in coordinate.values()), 40)
plt.figure(figsize=(6.5, 3.2))
plt.hist(coordinate["none"].numpy(), bins=bins, color=GRAY, alpha=0.7,
         label=f"no baseline (std {figure_data['std']['no_baseline']:.3f})")
plt.hist(coordinate["loo"].numpy(), bins=bins, color=ACCENT, alpha=0.7,
         label=f"leave-one-out (std {figure_data['std']['leave_one_out']:.3f})")
plt.axvline(exact_value, color="black", lw=1.2, label=f"exact gradient {exact_value:.4f}")
plt.xlabel(f"estimate of dJ/d theta{list(COORD)}")
plt.ylabel(f"batches (of {K})")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Checkpoint 2d: the ordering held at all five seeds measured (see the notes below).
workshop.checkpoint(2, label="2d")
assert total_variance["none"] > total_variance["loo"] > total_variance["token_loo"], total_variance
print("Checkpoint 2d passed: no baseline > leave-one-out > leave-one-out with token-level credit")

**Explain.** All estimators are centered on the exact gradient (2c), but the leave-one-out baseline removes about 40% of the variance, and token-level credit removes about 40% of what is left: a token is no longer blamed for mismatches at earlier positions, which it could not influence. The histogram shows one entry of $\hat g$: both clouds are centered on the vertical line, the baseline's is narrower. The batch-mean row is the leave-one-out row shrunk by the exact factor $(1 - 1/N)^2$ in variance, as the identity in 2b says it must be.

**Run: does less variance train faster?** Five seeds, 300 steps each, Adam at learning rate 0.05, with no baseline, with your leave-one-out baseline, and with the batch mean.

**Predict.** How many steps to reach $J \ge 0.9$ with and without the baseline? Will the batch mean behave like leave-one-out, or worse, since its gradient is smaller by $1/16$?

In [ ]:
# Run: five seeds with and without a baseline (about 5 to 15 s on a CPU).
SEEDS = [0, 1, 2, 3, 4]
started = time.time()
runs = {name: [train_toy(fn, seed=s)[1:] for s in SEEDS]
        for name, fn in [("no baseline", None), ("leave-one-out", loo_baseline),
                         ("batch mean", batch_mean_baseline)]}
seeds_seconds = time.time() - started
print(f"{'':<14}" + "".join(f"seed {s:<5}" for s in SEEDS) + "   (steps to J >= 0.9; final J)")
for name, results in runs.items():
    print(f"{name:<14}" + "".join(f"{str(first) if first is not None else '>300':<10}" for _, first in results)
          + "   final J " + ", ".join(f"{h[-1]:.3f}" for h, _ in results))
print(f"({seeds_seconds:.1f} s)")

plt.figure(figsize=(6.5, 3.2))
for name, color in (("no baseline", GRAY), ("leave-one-out", ACCENT)):
    for i, (history, _) in enumerate(runs[name]):
        plt.plot(history, color=color, lw=1, alpha=0.8, label=name if i == 0 else None)
plt.axhline(0.9, color="black", lw=0.8, ls=":")
plt.xlabel("step")
plt.ylabel("exact J")
plt.title("Five seeds, Adam at learning rate 0.05")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Checkpoint 2e
workshop.checkpoint(2, label="2e")
def _steps(results):  # a run that never reached 0.9 counts as 301 steps
    return [first if first is not None else 301 for _, first in results]


assert all(first is not None for _, first in runs["leave-one-out"]), "every baseline run reaches J >= 0.9"
assert np.mean(_steps(runs["leave-one-out"])) < np.mean(_steps(runs["no baseline"]))
print(f"Checkpoint 2e passed: mean steps to J >= 0.9: {np.mean(_steps(runs['leave-one-out'])):.0f} with the "
      f"baseline, {np.mean(_steps(runs['no baseline'])):.0f} without")
TIMES["Exercise 2"] = time.time() - LAB_START - sum(TIMES.values())

**Explain.** With the baseline every seed reaches $J \ge 0.9$ in roughly 100 steps, about twice as fast as without, and the curves are tighter. Same expected gradient, less noise per step, faster progress.

The batch mean gives the same step counts as leave-one-out, to the step. Its gradient is exactly $1 - 1/N$ times smaller on every batch, and Adam divides each step by a running estimate of the gradient's scale, so a constant factor cancels (up to Adam's small $\epsilon$). With plain gradient descent it would act like a learning rate smaller by $1/16$. That is why Lab 10 can use the batch mean.

<details>
<summary>Optional: noise can stall training (1 minute)</summary>

The briefing warned that a policy pushed hard by a noisy estimate can become nearly deterministic on a wrong choice, and then stops sampling the alternatives. The next cell trains the same five seeds at learning rate 0.1. Nothing is asserted; compare the final values of $J$.

</details>

In [ ]:
# Optional: learning rate 0.1, without and with the baseline. Printed only.
for name, fn in (("no baseline", None), ("leave-one-out", loo_baseline)):
    finals = [train_toy(fn, seed=s, lr=0.1)[1][-1] for s in SEEDS]
    print(f"lr 0.1, {name:<14} final J: " + ", ".join(f"{J:.3f}" for J in finals))
print("A final J near 0.83 = 5/6 means one position is locked on a wrong symbol.")

<details>
<summary>How the thresholds of Part A were set</summary>

Measured on a CPU (Linux x86_64 container, 2 PyTorch threads, `torch` 2.14.1), with the solution code, at seeds 0 to 4 unless stated:

- **Checkpoints 1b and 2c, cosine with the exact gradient** (mean of 500 batches of 16): no baseline 0.983 to 0.984; leave-one-out 0.989 to 0.992; token-level 0.994 to 0.996. Floor 0.95. At a mid-training policy the exact gradient is small and Monte Carlo means are far from it, so the check runs at the uniform policy only.
- **Checkpoint 1c, final $J$ without a baseline** after 300 steps: 0.910 to 0.978 (seeds 0 to 4); 0.895 to 0.978 over seeds 0 to 19, where seed 17 never reached 0.9. Floor 0.85.
- **Checkpoint 2d, total variance:** no baseline 0.0109 to 0.0115, leave-one-out 0.0062 to 0.0065, token-level 0.0036 to 0.0038 (ratios about 0.57 and 0.33). Only the ordering is asserted.
- **Checkpoint 2e, steps to $J \ge 0.9$:** with the baseline 90 to 107 (89 to 117 over seeds 0 to 19), without 148 to 295. The checkpoint asserts that every baseline seed reaches 0.9 and that the mean is lower with the baseline.

The samples depend on PyTorch's random number generator, so another PyTorch version may draw different episodes; the floors leave room for that.

</details>

# Part B · Learning a Reward From Comparisons

## The Data

Part A had a reward we could compute. For "helpful" or "polite" there is no program, but people can compare two responses (briefing section 6). Part B learns a reward from comparisons, on data where we know the answer.

- **Prompts.** 256 everyday sentence openings that we wrote, each exactly 8 GPT-2 tokens: one of 16 subjects ("My sister") followed by one of 16 frames ("opened the letter and felt so"). Every frame stops just before a feeling, so GPT-2's continuations often contain one. The frames are split three ways: 10 for training pairs (160 prompts), 2 for held-out pairs (32 prompts) and 4 for Lab 10's evaluation (64 prompts), so held-out and evaluation prompts use frames the reward model never trained on.
- **Responses.** Two 24-token continuations per prompt, sampled from the small pretrained GPT-2 of Lab 6 (`distilbert/distilgpt2`) at temperature 1. This is $\pi_{\text{ref}}$, the model Lab 10 fine-tunes. The file stores GPT-2's token ids and the decoded text of every response.
- **The rater.** A hidden **gold rule** $g$ scores every response, and a simulated rater prefers response $a$ to $b$ with the Bradley–Terry probability $\sigma\big((g_a - g_b) / \tau_{\text{label}}\big)$ (briefing eq. `bt-rater`), with $\tau_{\text{label}} = 0.5$.

**The gold rule is ours.** It is a rule we wrote so that we can check what the reward model learns, and we know it only because the data are synthetic. A real preference dataset has no gold score: the raters' reasons are unknown, and the reward model is all there is. The rule, in symbols:

$$
g(x, y) = \min(n_{\text{pos}}, 3) - n_{\text{neg}} - 10 \max(0, f_{\text{list}} - 0.25),
$$

where, over the complete words of the response, $n_{\text{pos}}$ counts the *distinct* words from a list of 20 positive words (at most three count), $n_{\text{neg}}$ counts *every* occurrence of a word from a list of 20 negative words, and $f_{\text{list}}$ is the share of words that are on either list. The last term penalizes a response crowded with list words. GPT-2's own samples almost never trigger the cap or the crowding term; remember that for Lab 10.

The next cell restates the gold rule and the constants that Lab 10 shares with this lab, unchanged.

In [ ]:
# Shared with Lab 10, restated verbatim there (briefs: "Lab 9 -> Lab 10 interface").
PROMPT_LEN = 8      # tokens in every prompt; the build script asserts it
RESPONSE_LEN = 24   # tokens in every response; end-of-text is never sampled
EOS_ID = 50256      # == tokenizer.eos_token_id == len(tokenizer) - 1; asserted on load

WORD = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)*")
GOLD = {
    "positive": ["good", "great", "happy", "love", "loved", "wonderful", "beautiful", "nice",
                 "best", "amazing", "perfect", "excellent", "glad", "fun", "enjoyed",
                 "delicious", "proud", "lovely", "fantastic", "excited"],
    "negative": ["bad", "sad", "terrible", "awful", "hate", "angry", "worst", "horrible",
                 "poor", "wrong", "sick", "afraid", "worried", "upset", "disappointed",
                 "pain", "ugly", "cried", "scared", "dead"],
    "cap": 3, "crowd_threshold": 0.25, "crowd_weight": 10.0, "tau_label": 0.5,
}

def complete_words(prompt, response):
    """Lowercased words that end inside the response. A first word that continues the
    prompt's last word is joined to it; a word that touches the end may have been cut
    by the length limit, so it is dropped."""
    text = prompt + response
    return [m.group().lower() for m in WORD.finditer(text) if len(prompt) < m.end() < len(text)]

def gold_reward(prompt, response, gold=GOLD):
    words = complete_words(prompt, response)
    if not words:
        return 0.0
    pos, neg = set(gold["positive"]), set(gold["negative"])
    n_pos = len({w for w in words if w in pos})             # distinct positive words
    n_neg = sum(w in neg for w in words)                    # every negative occurrence
    f_list = sum(w in pos or w in neg for w in words) / len(words)
    return float(min(n_pos, gold["cap"]) - n_neg
                 - gold["crowd_weight"] * max(0.0, f_list - gold["crowd_threshold"]))


CAUSAL_LM = "distilbert/distilgpt2"  # models.causal_lm in the workshop's _variables.yml

_p = "My sister opened the old letter and felt"
for _y in (" so happy and proud of her, and a little worried about the future.", " great" * 24):
    print(f"g = {gold_reward(_p, _y):+.2f}  {_p!r} + {_y[:60]!r}")

In [ ]:
#@title Test-only: offline stand-in data (does nothing unless NLP_LLMS_OFFLINE_TINY=1) { display-mode: "form" }
# Builds files in the same schema as the real ones from a word-level stand-in: every word is one
# "token" with an id from a local vocabulary (inside GPT-2's id range), and responses are 24
# tokens drawn independently from a fixed distribution. NOT GPT-2 text; numbers computed from
# these files are not results. Used only to execute the notebook where the Hub is unreachable.
def build_offline_stand_in(folder, seed=0, n_train=8000, n_heldout=1000):
    folder.mkdir(exist_ok=True)
    gen = torch.Generator().manual_seed(seed)
    rng = random.Random(seed)
    subjects = ["My mother", "My father", "My sister", "My brother", "My friend", "My boss",
                "Our neighbor", "The teacher", "The doctor", "The manager", "The waiter",
                "The student", "The driver", "The coach", "The nurse", "The chef"]
    frames = {
        "train": ["looked at the results feeling so", "opened the letter and felt so",
                  "walked into the room feeling so", "tasted the soup and felt so",
                  "read the review and felt so", "came home late and felt so",
                  "listened to the song feeling so", "looked out the window feeling so",
                  "heard the news and felt so", "finished the long day feeling so"],
        "heldout": ["watched the game and felt so", "visited the old house feeling so"],
        "eval": ["left the new restaurant feeling so", "read the email and felt so",
                 "saw the bill and felt so", "spent the whole weekend feeling so"]}
    filler = ("the a and to of it was I we they he she that in on for with at but so not then "
              "all just very really one day time home back again here there out up about what "
              "when this my our his her their you me us them some more much had have has did "
              "said told thought felt saw found went came made got took still never always only "
              "even also well now later today morning night room door table window letter "
              "food work school car street city people kids little old new long last first big "
              "small next other same few many quiet slowly quickly finally almost maybe after "
              "before into over through around away down off like than as if or no yes").split()
    words = sorted(set(filler) | set(GOLD["positive"]) | set(GOLD["negative"])
                   | {w for t in subjects + sum(frames.values(), []) for w in t.split()})
    tokens = [" " + w for w in words] + words + [",", ".", "!"]
    vocab = {tok: 1000 + 3 * i for i, tok in enumerate(tokens)}
    inverse = {i: tok for tok, i in vocab.items()}
    encode = lambda text: [vocab[t] for t in re.findall(r" ?[A-Za-z']+|[,.!]", text)]
    decode = lambda ids: "".join(inverse[i] for i in ids)
    weights = {" " + w: 0.78 / (rank + 5) / 3.9 for rank, w in enumerate(rng.sample(filler, len(filler)))}
    for i, w in enumerate(GOLD["positive"] + GOLD["negative"]):
        weights[" " + w] = 0.02 / (1 + i % 5) / 9.12
    weights.update({",": 0.07, ".": 0.06, "!": 0.01})
    response_ids = torch.tensor([vocab[t] for t in weights])
    probs = torch.tensor(list(weights.values()), dtype=torch.float64)
    prompts = {s: [{"text": f"{a} {f}", "ids": encode(f"{a} {f}")} for f in fs for a in subjects]
               for s, fs in frames.items()}
    lines = []
    for split, n in (("train", n_train), ("heldout", n_heldout)):
        order = rng.sample(prompts[split], len(prompts[split]))
        draws = response_ids[torch.multinomial(probs, 2 * n * RESPONSE_LEN, True, generator=gen)]
        draws = draws.reshape(n, 2, RESPONSE_LEN).tolist()
        for i in range(n):
            p = order[i % len(order)]
            (a, b), texts = draws[i], [decode(draws[i][0]), decode(draws[i][1])]
            g = [gold_reward(p["text"], t) for t in texts]
            win = 0 if rng.random() < 1 / (1 + math.exp(-(g[0] - g[1]) / GOLD["tau_label"])) else 1
            lines.append(json.dumps({
                "prompt": p["text"], "prompt_ids": p["ids"], "chosen": texts[win],
                "rejected": texts[1 - win], "chosen_ids": draws[i][win],
                "rejected_ids": draws[i][1 - win], "gold_chosen": g[win],
                "gold_rejected": g[1 - win], "split": split}, separators=(",", ":")))
    (folder / "lab09_preferences.jsonl.gz").write_bytes(
        gzip.compress(("\n".join(lines) + "\n").encode("utf-8"), mtime=0))
    preview_texts = [("one positive word repeated", " great" * 24),
                     ("distinct positive words, comma-separated",
                      " good, great, happy, wonderful, nice, perfect, excellent."),
                     ("'I love it!' repeated", " I love it!" * 6),
                     ("ordinary reply, one positive word", " it was a good day, and we all went home."),
                     ("ordinary reply, two negative words", " it was a bad day, and we all felt sick.")]
    doc = {"format": "nlp-llms/lab09-prompts", "version": 1, "model": "STAND-IN (not GPT-2)",
           "revision": "none", "subjects": subjects, "frames": frames, "prompts": prompts,
           "preview": [{"name": n, "response": t, "response_ids": encode(t)[:RESPONSE_LEN]}
                       for n, t in preview_texts],
           "gold": GOLD, "build": {"stand_in": True, "seed": seed}}
    (folder / "lab09_prompts.json").write_text(json.dumps(doc), encoding="utf-8")


if OFFLINE_TINY:
    build_offline_stand_in(Path("offline-tiny"))
    print("Built the OFFLINE STAND-IN data in offline-tiny/. Not GPT-2 samples; not results.")

The next two cells load the data. The first is the workshop's standard loading cell, the same in every lab (`data/README.md` in the repository): it checks each file against a SHA-256 hash, so everyone gets the same bytes. The second loads this lab's two files and checks them against the interface Lab 10 relies on.

In [ ]:
import csv, gzip, hashlib, io, os, urllib.request
from pathlib import Path


def fetch(name, urls, sha256):
    """Return the bytes of a workshop data file, checked against its SHA-256."""
    cache = Path(os.environ.get("NLP_LLMS_DATA", "data"))
    for path in (cache / name, Path("../data") / name):
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == sha256:
            return path.read_bytes()
    for url in urls:  # canonical URL first, fallback second
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                blob = response.read()
        except OSError as error:
            print(f"Could not fetch {url}: {error}")
            continue
        if hashlib.sha256(blob).hexdigest() != sha256:
            print(f"Ignoring {url}: the file does not match the expected hash")
            continue
        cache.mkdir(parents=True, exist_ok=True)
        (cache / name).write_bytes(blob)
        return blob
    raise RuntimeError(f"Could not load {name} from any source")


TOPIC_CLASSES = ["cs.CL", "cs.CV", "cs.LG", "cs.RO"]  # label 0, 1, 2, 3
TOPIC_NAMES = ["Language", "Vision", "Machine learning", "Robotics"]


def load_topics():
    """arXiv Topics v1. Returns {"train" | "val" | "test": (texts, labels)}."""
    blob = fetch(
        "arxiv_topics_v1.csv.gz",
        [
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/arxiv_topics_v1.csv.gz",
            "https://cdn.jsdelivr.net/gh/project-delphi/nlp-llms@main/data/arxiv_topics_v1.csv.gz",
        ],
        "49b51502fc6cf6ecff23fa266c5372594a88414dd96513579237792710246dc5",
    )
    splits = {"train": ([], []), "val": ([], []), "test": ([], [])}
    for row in csv.DictReader(io.StringIO(gzip.decompress(blob).decode("utf-8"))):
        texts, labels = splits[row["split"]]
        texts.append(row["title"] + "\n" + row["abstract"])
        labels.append(int(row["label"]))
    return splits


def load_lm_corpus():
    """Tiny Shakespeare. Returns {"train" | "val" | "test": str}, split by character offset."""
    blob = fetch(
        "tinyshakespeare.txt",
        [
            "https://raw.githubusercontent.com/karpathy/char-rnn/6f9487a6fe5b420b7ca9afb0d7c078e37c1d1b4e/data/tinyshakespeare/input.txt",
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/tinyshakespeare.txt",
        ],
        "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",
    )
    text = blob.decode("utf-8")
    return {"train": text[:1_000_000], "val": text[1_000_000:1_055_000], "test": text[1_055_000:]}

In [ ]:
# Lab 9's files. The hashes are printed by data/build_lab09_preferences.py and recorded in
# _variables.yml (datasets lab09_prompts and lab09_preferences). Built 2026-10-06.
LAB09_FILES = {
    "prompts": {
        "name": "lab09_prompts.json",
        "urls": ["https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/lab09_prompts.json",
                 "https://cdn.jsdelivr.net/gh/project-delphi/nlp-llms@main/data/lab09_prompts.json"],
        "sha256": "966971fd2f9f8533711db4cd2f8eaf36e839bd0fb9ed1c1c52efc970f6a25ed6",
    },
    "preferences": {
        "name": "lab09_preferences.jsonl.gz",
        "urls": ["https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/lab09_preferences.jsonl.gz",
                 "https://cdn.jsdelivr.net/gh/project-delphi/nlp-llms@main/data/lab09_preferences.jsonl.gz"],
        "sha256": "c00b775865dabf491542719b223ab6fb539f3728ca0348c8ca50ecb51b752326",
    },
}


def load_lab09_file(key):
    """The bytes of one of Lab 9's files: the stand-in in test mode, else through fetch()."""
    entry = LAB09_FILES[key]
    if OFFLINE_TINY:
        return (Path("offline-tiny") / entry["name"]).read_bytes()
    if entry["sha256"] is not None:
        return fetch(entry["name"], entry["urls"], entry["sha256"])
    cache = Path(os.environ.get("NLP_LLMS_DATA", "data"))
    for path in (cache / entry["name"], Path("../data") / entry["name"]):
        if path.exists():
            print(f"WARNING: {entry['name']} has no pinned hash yet; using {path} unchecked")
            return path.read_bytes()
    raise RuntimeError(
        f"{entry['name']} has not been built yet. It is built by data/build_lab09_preferences.py "
        "on a machine with access to the Hugging Face Hub (see the script's docstring).")


blobs = {key: load_lab09_file(key) for key in LAB09_FILES}
data_sha256 = {key: hashlib.sha256(blob).hexdigest() for key, blob in blobs.items()}
prompts_doc = json.loads(blobs["prompts"].decode("utf-8"))
pairs = [json.loads(line) for line in gzip.decompress(blobs["preferences"]).decode("utf-8").splitlines()]

# The interface checks: format, lengths, no end-of-text, gold scores recomputed from the text.
assert prompts_doc["format"] == "nlp-llms/lab09-prompts" and prompts_doc["version"] == 1
assert prompts_doc.get("gold", GOLD) == GOLD, "GOLD in this notebook differs from the data's"
assert [len(prompts_doc["prompts"][s]) for s in ("train", "heldout", "eval")] == [160, 32, 64]
for p in pairs:
    assert len(p["prompt_ids"]) == PROMPT_LEN, p["prompt"]
    assert len(p["chosen_ids"]) == len(p["rejected_ids"]) == RESPONSE_LEN
    assert EOS_ID not in p["chosen_ids"] + p["rejected_ids"]
    assert gold_reward(p["prompt"], p["chosen"]) == p["gold_chosen"]
    assert gold_reward(p["prompt"], p["rejected"]) == p["gold_rejected"]


def pair_tensors(split):
    """The pairs of one split as tensors (and their texts)."""
    rows = [p for p in pairs if p["split"] == split]
    return {
        "prompt_ids": torch.tensor([p["prompt_ids"] for p in rows]),      # (n, 8) GPT-2 ids
        "chosen_ids": torch.tensor([p["chosen_ids"] for p in rows]),      # (n, 24)
        "rejected_ids": torch.tensor([p["rejected_ids"] for p in rows]),  # (n, 24)
        "gold_chosen": torch.tensor([p["gold_chosen"] for p in rows]),    # (n,) float
        "gold_rejected": torch.tensor([p["gold_rejected"] for p in rows]),
        "rows": rows,
    }


train, heldout = pair_tensors("train"), pair_tensors("heldout")
heldout_prompts = {p["text"] for p in prompts_doc["prompts"]["heldout"]}
assert {p["prompt"] for p in heldout["rows"]} <= heldout_prompts, "held-out pairs use held-out prompts"
assert not {p["prompt"] for p in train["rows"]} & heldout_prompts
print(f"Policy that wrote the responses: {prompts_doc['model']} @ {prompts_doc['revision']}")
print(f"{len(train['rows']):,} train pairs, {len(heldout['rows']):,} held-out pairs; "
      f"files: {', '.join(f'{k} {v[:12]}...' for k, v in data_sha256.items())}")
if OFFLINE_TINY:
    print("OFFLINE STAND-IN DATA: the numbers in Part B are not results.")

**Read three pairs.** The next cell prints three training pairs: the prompt, both responses, their gold scores and which one the rater chose. Look for the words that moved the gold score, and for a pair where the rater chose the response with the *lower* score.

In [ ]:
def show_pair(p):
    print(f"PROMPT   {p['prompt']!r}")
    print(f"chosen   g = {p['gold_chosen']:+.2f}  {p['chosen']!r}")
    print(f"rejected g = {p['gold_rejected']:+.2f}  {p['rejected']!r}\n")


_untied = [p for p in train["rows"] if p["gold_chosen"] != p["gold_rejected"]]
_upset = next(p for p in train["rows"] if p["gold_chosen"] < p["gold_rejected"])
for p in (_untied[0], _untied[1], _upset):
    show_pair(p)

## Exercise 3 · The Bradley–Terry Rater (8 minutes)

Bradley–Terry (briefing eqs. `bt` and `bt-rater`) says response $a$ is preferred to $b$ with probability

$$
P(y_a \succ y_b \mid x) = \sigma\Big(\frac{g_a - g_b}{\tau_{\text{label}}}\Big),
$$

here with the gold score as the latent quality. A low $\tau_{\text{label}}$ is a consistent rater; as $\tau_{\text{label}} \to \infty$ the rater flips a coin.

**Predict.** With $\tau_{\text{label}} = 0.5$: how often does a response with $g = 1$ beat one with $g = 0$? Two responses with equal scores?

**Task.** Complete `bt_prob(g_a, g_b, tau_label)`. It must work on Python floats and on tensors (use `torch.as_tensor` and `torch.sigmoid`, and return a float when both inputs are floats).

In [ ]:
# TODO 3: complete this function.
def bt_prob(g_a, g_b, tau_label):
    """sigmoid((g_a - g_b) / tau_label): P(a preferred to b). Floats or tensors."""
    raise NotImplementedError("TODO 3")

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def bt_prob(g_a, g_b, tau_label):
    """sigmoid((g_a - g_b) / tau_label): P(a preferred to b). Floats or tensors."""
    p = torch.sigmoid((torch.as_tensor(g_a, dtype=torch.float64) - g_b) / tau_label)
    if isinstance(g_a, (int, float)) and isinstance(g_b, (int, float)):
        return p.item()  # a Python float for float inputs
    return p.to(torch.get_default_dtype())

<details>
<summary>Why this solution works</summary>

Dividing the gap by $\tau_{\text{label}}$ before the sigmoid is exactly how a sampling temperature divides logits (Module 3): a small temperature makes the probability close to 0 or 1, a large one pushes it toward $1/2$. Since $\sigma(-z) = 1 - \sigma(z)$, `bt_prob(a, b) + bt_prob(b, a) = 1`: one of the two responses is always chosen. Only the difference $g_a - g_b$ enters, which is the "only differences" property of briefing section 7: adding a constant to every score changes no probability.

</details>

In [ ]:
# Checkpoint 3a: hand values.
workshop.checkpoint(3, label="3a")
assert abs(bt_prob(1.0, 0.0, 0.5) - 1 / (1 + math.exp(-2))) < 1e-9, (
    f"bt_prob(1, 0, 0.5) must be sigmoid((1 - 0) / 0.5) = {1 / (1 + math.exp(-2)):.10f}, got {bt_prob(1.0, 0.0, 0.5)}: "
    "divide the gap by tau_label, and compute in float64 (the tolerance is 1e-9)")
assert abs(bt_prob(0.3, 0.3, 2.0) - 0.5) < 1e-12, f"equal scores give 0.5, got {bt_prob(0.3, 0.3, 2.0)}"
assert abs(bt_prob(2.0, -1.0, 1.0) + bt_prob(-1.0, 2.0, 1.0) - 1) < 1e-12, (
    f"P(a preferred to b) + P(b preferred to a) must be 1, got {bt_prob(2.0, -1.0, 1.0) + bt_prob(-1.0, 2.0, 1.0)}")
_ga, _gb = torch.tensor([1.0, 0.0, -2.0]), torch.tensor([0.0, 0.0, 1.0])
assert torch.allclose(bt_prob(_ga, _gb, 0.5), torch.sigmoid((_ga - _gb) / 0.5)), "works on tensors"
assert torch.allclose(bt_prob(_ga + 7.0, _gb + 7.0, 0.5), bt_prob(_ga, _gb, 0.5)), "only differences"
print(f"Checkpoint 3a passed: bt_prob(1, 0, 0.5) = {bt_prob(1.0, 0.0, 0.5):.4f}")

# Checkpoint 3b: do the labels in the file follow your bt_prob? For untied training pairs, bin by
# the gap |g_a - g_b| and compare the share of pairs where the higher-scored response was chosen
# with the mean of your predicted probability. Tolerance: 3 standard errors of a share, plus 0.01.
TAU_LABEL = GOLD["tau_label"]
g_c, g_r = train["gold_chosen"], train["gold_rejected"]
gap = (g_c - g_r).abs()
untied = gap > 1e-9
higher_chosen = (g_c > g_r).float()
predicted = bt_prob(torch.maximum(g_c, g_r), torch.minimum(g_c, g_r), TAU_LABEL)
edges = [0.0, 0.75, 1.25, 1.75, 2.5, float("inf")]
print(f"\n{'gap |g_a - g_b|':<16} {'pairs':>6} {'observed':>9} {'bt_prob':>8} {'tolerance':>10}")
for lo, hi in zip(edges[:-1], edges[1:]):
    in_bin = untied & (gap > lo) & (gap <= hi)
    n = int(in_bin.sum())
    if n == 0:
        continue
    observed, expected = higher_chosen[in_bin].mean().item(), predicted[in_bin].mean().item()
    tolerance = 3 * math.sqrt(max((predicted[in_bin] * (1 - predicted[in_bin])).mean().item(), 1e-4) / n) + 0.01
    flag = "" if n >= 50 else "  (fewer than 50 pairs: not checked)"
    print(f"({lo:.2f}, {hi:.2f}]{'':<5} {n:>6} {observed:>9.3f} {expected:>8.3f} {tolerance:>10.3f}{flag}")
    if n >= 50:
        assert abs(observed - expected) <= tolerance, (lo, hi, observed, expected)
print("Checkpoint 3b passed: the file's labels follow your Bradley-Terry probabilities")

**Run: the ceiling.** Even the true gold score predicts a held-out label only with probability $\max(p, 1 - p)$, because the rater is random (briefing eq. `oracle`):

$$
\mathrm{Acc}^\star = \mathbb{E}_{\text{pairs}}\big[\max(p, 1 - p)\big], \qquad p = \sigma\Big(\frac{g_a - g_b}{\tau_{\text{label}}}\Big).
$$

A tied pair contributes $1/2$. The provided `oracle_accuracy` computes it on the held-out pairs: no reward model can beat it on those labels, except by chance.

**Predict.** Will $\mathrm{Acc}^\star$ be closer to 0.6 or to 0.95? What share of pairs do you expect to be ties?

In [ ]:
def oracle_accuracy(g_a, g_b, tau_label):
    """Acc* = mean of max(p, 1 - p), p = bt_prob(g_a, g_b, tau_label) (briefing eq. oracle)."""
    p = bt_prob(torch.as_tensor(g_a), torch.as_tensor(g_b), tau_label)
    return torch.maximum(p, 1 - p).mean().item()


h_c, h_r = heldout["gold_chosen"], heldout["gold_rejected"]
tie_share = (h_c == h_r).float().mean().item()
oracle_acc = oracle_accuracy(h_c, h_r, TAU_LABEL)
upsets = ((h_c < h_r).float().mean().item())
print(f"held-out pairs: {len(h_c):,} · ties {tie_share:.3f} · the rater chose the lower-scored response "
      f"in {upsets:.3f}")
print(f"Acc* on held-out pairs at tau_label = {TAU_LABEL}: {oracle_acc:.4f}  <- the ceiling for Exercise 4")
TIMES["Exercise 3"] = time.time() - LAB_START - sum(TIMES.values())

**Explain.** Ties are common: a continuation with no list word scores 0, and two such continuations tie. Ties and close pairs pull $\mathrm{Acc}^\star$ well below 1. A reward model is judged against this ceiling, not against 100%. The rater also sometimes chose the lower-scored response: that is the label noise of eq. `bt-rater`, and no model should try to fit it.

## Exercise 4 · The Reward-Model Loss, and Training (12 minutes)

Replace the gold score by a model $r_\phi$ and maximize the likelihood of the observed comparisons (briefing eq. `rm-loss`):

$$
\mathcal{L}_{\text{RM}}(\phi) = -\frac{1}{|\mathcal{D}_{\text{pref}}|} \sum_{(x,\, y_w,\, y_l)} \log \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big).
$$

It is the binary cross-entropy of logistic regression with logit $r_\phi(x, y_w) - r_\phi(x, y_l)$ and label "the first one wins". $\tau_{\text{label}}$ does not appear: it is absorbed into the scale of $r_\phi$.

**Predict.** What is the loss when the model gives both responses the same score? What happens to the loss if you add 5 to every score?

**Task.** Complete `bt_loss(reward_w, reward_l)` for two tensors of shape `(n,)`. Use `F.logsigmoid`, not `torch.log(torch.sigmoid(...))`.

<details>
<summary>Hint for TODO 4</summary>

The chosen response's score comes first: the logit is $r_w - r_l$, and the loss is its negative log-sigmoid averaged over the pairs (briefing eq. `rm-loss`). With the order swapped, equal scores still give $\log 2$ but every other pair is scored backwards, which the comparison with the logistic-regression loss catches. Use `F.logsigmoid`: at a gap of $-400$, `torch.sigmoid` underflows to 0 and its log is $-\infty$.

</details>

In [ ]:
# TODO 4: complete this function.
def bt_loss(reward_w, reward_l):
    """-log sigmoid(reward_w - reward_l), averaged. reward_w, reward_l: (n,) -> scalar."""
    raise NotImplementedError("TODO 4")

In [ ]:
#@title Solution 4 — try it yourself first { display-mode: "form" }
@workshop.solution(4)
def bt_loss(reward_w, reward_l):
    """-log sigmoid(reward_w - reward_l), averaged. reward_w, reward_l: (n,) -> scalar."""
    return -F.logsigmoid(reward_w - reward_l).mean()

<details>
<summary>Why this solution works</summary>

$\sigma(r_w - r_l)$ is the model's Bradley–Terry probability that the chosen response wins, so the loss is the mean negative log-likelihood of the observed labels. `F.logsigmoid(z)` computes $\log \sigma(z) = -\log(1 + e^{-z})$ in a stable way: for a large negative $z$, `torch.sigmoid` would underflow to 0 and `log` would return $-\infty$. Equal scores give $-\log \frac12 = \log 2$. Adding a constant to both scores leaves the difference, and so the loss, unchanged: the reward model's level is arbitrary, which is why Exercise 5 fixes it. Lab 10's DPO loss calls this same function with a different reward.

</details>

In [ ]:
# Checkpoint 4a
workshop.checkpoint(4, label="4a")
_zero = torch.zeros(4)
assert abs(bt_loss(_zero, _zero).item() - math.log(2)) < 1e-7, (
    f"equal scores: the loss must be log 2 = {math.log(2):.6f}, got {bt_loss(_zero, _zero).item()}: "
    "average -log sigmoid(r_w - r_l) over the pairs, with the natural log")
_w, _l = torch.tensor([2.0, -1.0, 0.5, 30.0]), torch.tensor([1.0, 1.0, 0.5, -30.0])
_reference = F.binary_cross_entropy_with_logits(_w - _l, torch.ones(4))
assert torch.allclose(bt_loss(_w, _l), _reference), (
    f"the loss {bt_loss(_w, _l).item():.4f} differs from the logistic-regression BCE on r_w - r_l, "
    f"{_reference.item():.4f}: the chosen response's score comes first")
assert torch.allclose(bt_loss(_w + 5.0, _l + 5.0), bt_loss(_w, _l)), "a constant shift changes nothing"
assert torch.isfinite(bt_loss(torch.tensor([-200.0]), torch.tensor([200.0]))), "use F.logsigmoid"
print(f"Checkpoint 4a passed: bt_loss(0, 0) = {bt_loss(_zero, _zero).item():.6f} = log 2")

**The reward model (provided).** A small transformer built from the block of Lab 5 (pre-norm, causal mask, dropout off). It reads the GPT-2 token ids of prompt and response together, 8 + 24 = 32 positions, and a linear head reads one number from the last position, which has seen everything.

- **From scratch, on purpose.** Real reward models start from a pretrained language model, often the SFT model, with a scalar head (briefing section 8). This one is trained from scratch, for two reasons: it trains on a CPU in a couple of minutes with no download, and its blind spots are predictable, which Lab 10 needs.
- **A compact vocabulary.** GPT-2 has 50,257 tokens, most of which never occur in our pairs. `vocab_map` gives each token id that occurs at least twice in the training pairs its own embedding row, numbered from 1; every other id shares row 0. A token the reward model never saw looks like every other unseen token.
- `raw(prompt_ids, response_ids)` returns $r_\phi$, shape `(N,)`. `forward` returns the normalized score of Exercise 5; until then the normalization is the identity.

The class and its attribute names are fixed by the interface with Lab 10, which loads the saved weights into the same class.

In [ ]:
# Lab 5's attention, mask and block, restated (the solutions of Lab 5's Exercises 1 and 2).
def scaled_dot_product_attention(Q, K, V, mask=None):
    """softmax(Q K^T / sqrt(d_k)) V on the last two dimensions; any leading dimensions."""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (..., T, T)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    A = torch.softmax(scores, dim=-1)
    return A @ V, A


def causal_mask(T):
    """(T, T) bool tensor, True where position t may read position i, that is where i <= t."""
    return torch.tril(torch.ones(T, T, dtype=torch.bool))


class MultiHeadAttention(nn.Module):
    """Masked multi-head self-attention (Lab 5)."""

    def __init__(self, d, n_h, dropout=0.0):
        super().__init__()
        assert d % n_h == 0, "d must be divisible by n_h"
        self.n_h, self.d_k = n_h, d // n_h
        self.W_Q = nn.Linear(d, d, bias=False)
        self.W_K = nn.Linear(d, d, bias=False)
        self.W_V = nn.Linear(d, d, bias=False)
        self.proj = nn.Linear(d, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, H, mask):
        B, T, d = H.shape
        Q = self.W_Q(H).view(B, T, self.n_h, self.d_k).transpose(1, 2)  # (B, n_h, T, d_k)
        K = self.W_K(H).view(B, T, self.n_h, self.d_k).transpose(1, 2)
        V = self.W_V(H).view(B, T, self.n_h, self.d_k).transpose(1, 2)
        V_bar, A = scaled_dot_product_attention(Q, K, V, mask)
        V_bar = V_bar.transpose(1, 2).reshape(B, T, d)  # concatenate the heads
        return self.dropout(self.proj(V_bar)), A


class Block(nn.Module):
    """One pre-norm transformer block (Lab 5)."""

    def __init__(self, d, n_h, d_ff, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, n_h, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d), nn.Dropout(dropout))

    def forward(self, H, mask):
        attn_out, A = self.attn(self.ln1(H), mask)
        H = H + attn_out  # H' = H + MHA(LN(H))
        H = H + self.ffn(self.ln2(H))  # H_out = H' + FFN(LN(H'))
        return H, A


class RewardModel(nn.Module):
    """Small transformer trained from scratch on GPT-2 token ids: Lab 5's pre-norm block
    with a causal mask; a scalar head reads the last position of prompt + response."""

    def __init__(self, n_rows, d=64, n_h=4, n_l=2, T_max=32, d_ff=256):
        super().__init__()
        self.T_max = T_max
        # GPT-2 id -> embedding row; row 0 is shared by every id not seen twice in the train pairs.
        self.register_buffer("vocab_map", torch.zeros(EOS_ID + 1, dtype=torch.long))
        self.register_buffer("score_mean", torch.tensor(0.0))  # set by Exercise 5
        self.register_buffer("score_std", torch.tensor(1.0))
        self.tok_emb = nn.Embedding(n_rows, d)
        self.pos_emb = nn.Embedding(T_max, d)
        self.blocks = nn.ModuleList([Block(d, n_h, d_ff, dropout=0.0) for _ in range(n_l)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, 1)
        for module in self.modules():  # GPT-2 style: small normal weights, zero biases (Lab 5)
            if isinstance(module, (nn.Linear, nn.Embedding)):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def raw(self, prompt_ids, response_ids):
        """(N, P), (N, L) int64 GPT-2 ids -> (N,) raw r_phi."""
        ids = torch.cat([prompt_ids, response_ids], dim=1)  # (N, P + L)
        T = ids.shape[1]
        assert T <= self.T_max, f"prompt + response must be at most {self.T_max} tokens"
        H = self.tok_emb(self.vocab_map[ids]) + self.pos_emb(torch.arange(T, device=ids.device))
        mask = causal_mask(T).to(ids.device)
        for block in self.blocks:
            H, _ = block(H, mask)
        return self.head(self.ln_f(H[:, -1]))[:, 0]  # the last position has read everything

    def forward(self, prompt_ids, response_ids):
        """(N,) normalized: (raw - score_mean) / score_std."""
        return (self.raw(prompt_ids, response_ids) - self.score_mean) / self.score_std


def build_vocab_map(data, min_count=2):
    """Each GPT-2 id seen at least min_count times in the pairs gets a row, from 1 in id order."""
    ids = torch.cat([data["prompt_ids"].reshape(-1), data["chosen_ids"].reshape(-1),
                     data["rejected_ids"].reshape(-1)])
    counts = torch.bincount(ids, minlength=EOS_ID + 1)
    kept = (counts >= min_count).nonzero()[:, 0]  # increasing id order
    vocab_map = torch.zeros(EOS_ID + 1, dtype=torch.long)
    vocab_map[kept] = torch.arange(1, len(kept) + 1)
    return vocab_map, len(kept) + 1


RM_CONFIG = {"d": 64, "n_h": 4, "n_l": 2, "T_max": 32, "d_ff": 256}
vocab_map, n_rows = build_vocab_map(train)
_responses = torch.cat([heldout["chosen_ids"], heldout["rejected_ids"]])
print(f"n_rows = {n_rows:,} embedding rows ({n_rows - 1:,} token ids seen at least twice in the train pairs)")
print(f"held-out response tokens that map to the shared row 0: {(vocab_map[_responses] == 0).float().mean().item():.1%}")
_model = RewardModel(n_rows, **RM_CONFIG)
print(f"RewardModel parameters: {sum(p.numel() for p in _model.parameters()):,} "
      f"(of which the embedding table {n_rows * RM_CONFIG['d']:,})")

**Run: train the reward model.** The provided loop trains with your `bt_loss`: AdamW, batches of 64 pairs, a few epochs over the training pairs. The chosen and rejected responses of a batch go through the model in one call. The last 800 training pairs are kept aside as a validation slice, and the epoch with the best pairwise accuracy on that slice is kept. The held-out pairs are not touched until the evaluation.

**Predict.** Your held-out pairwise accuracy will be close to which number printed in Exercise 3?

In [ ]:
RM_TRAIN = {"epochs": 6, "batch_size": 64, "lr": 2e-3, "weight_decay": 0.01, "val_pairs": 800}
RM_SEED = int(os.environ.get("NLP_LLMS_RM_SEED", SEED))  # the build uses seeds 0, 1 and 2


@torch.no_grad()
def score(model, prompt_ids, response_ids, batch_size=1024, normalized=False):
    """r_phi for many responses, (N,), on the CPU."""
    model.eval()
    fn, device = (model if normalized else model.raw), next(model.parameters()).device
    out = [fn(prompt_ids[i:i + batch_size].to(device), response_ids[i:i + batch_size].to(device)).cpu()
           for i in range(0, len(prompt_ids), batch_size)]
    return torch.cat(out)


def pairwise_accuracy(model, data, idx=None):
    idx = torch.arange(len(data["prompt_ids"])) if idx is None else idx
    r_w = score(model, data["prompt_ids"][idx], data["chosen_ids"][idx])
    r_l = score(model, data["prompt_ids"][idx], data["rejected_ids"][idx])
    return (r_w > r_l).float().mean().item()


def train_reward_model(data, vocab_map, n_rows, seed=RM_SEED, epochs=6, batch_size=64, lr=2e-3,
                       weight_decay=0.01, val_pairs=800, verbose=True):
    """Train RewardModel with bt_loss; keep the epoch with the best validation-slice accuracy."""
    torch.manual_seed(seed)
    model = RewardModel(n_rows, **RM_CONFIG)
    model.vocab_map.copy_(vocab_map)
    model.to(DEVICE)
    n = len(data["prompt_ids"])
    fit_idx, val_idx = torch.arange(n - val_pairs), torch.arange(n - val_pairs, n)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    generator = torch.Generator().manual_seed(seed)
    best, history = (-1.0, None, 0), []
    for epoch in range(1, epochs + 1):
        model.train()
        order = fit_idx[torch.randperm(len(fit_idx), generator=generator)]
        losses = []
        for start in range(0, len(order), batch_size):
            b = order[start:start + batch_size]
            prompts = data["prompt_ids"][b].repeat(2, 1).to(DEVICE)  # (2B, 8)
            responses = torch.cat([data["chosen_ids"][b], data["rejected_ids"][b]]).to(DEVICE)
            reward_w, reward_l = model.raw(prompts, responses).chunk(2)  # (B,) each
            loss = bt_loss(reward_w, reward_l)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        val_acc = pairwise_accuracy(model, data, val_idx) if val_pairs else float("nan")
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_acc": val_acc})
        if verbose:
            print(f"epoch {epoch}: train loss {np.mean(losses):.4f} · validation-slice accuracy {val_acc:.4f}")
        if not val_pairs or val_acc > best[0]:
            best = (val_acc, copy.deepcopy(model.state_dict()), epoch)
    model.load_state_dict(best[1])
    return model.cpu().eval(), history, best[2]


started = time.time()
reward_model, rm_history, best_epoch = train_reward_model(train, vocab_map, n_rows, **RM_TRAIN)
rm_seconds = time.time() - started
print(f"Trained in {rm_seconds:.0f} s on {DEVICE} (seed {RM_SEED}); kept epoch {best_epoch}")
TIMES["Exercise 4 training"] = rm_seconds

In [ ]:
# Evaluation on the held-out pairs: frames the reward model never trained on.
def average_ranks(x):
    """Ranks 0..n-1 with ties given their average rank (for Spearman's correlation)."""
    x = np.asarray(x, dtype=np.float64)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x))
    ranks[order] = np.arange(len(x))
    _, inverse, counts = np.unique(x, return_inverse=True, return_counts=True)
    return (np.bincount(inverse, weights=ranks) / counts)[inverse]


def spearman(x, y):
    return float(np.corrcoef(average_ranks(x), average_ranks(y))[0, 1])


def evaluate_reward_model(model, data, tau_label=TAU_LABEL):
    r_w = score(model, data["prompt_ids"], data["chosen_ids"])
    r_l = score(model, data["prompt_ids"], data["rejected_ids"])
    g_w, g_l = data["gold_chosen"], data["gold_rejected"]
    untied = (g_w != g_l)
    return {
        "heldout_acc": (r_w > r_l).float().mean().item(),
        "oracle_acc": oracle_accuracy(g_w, g_l, tau_label),
        "acc_gold_order_untied": ((r_w > r_l) == (g_w > g_l))[untied].float().mean().item(),
        "spearman_gold": spearman(torch.cat([r_w, r_l]), torch.cat([g_w, g_l])),
    }


metrics = evaluate_reward_model(reward_model, heldout)
print(f"held-out pairwise accuracy (against the rater's labels): {metrics['heldout_acc']:.4f}")
print(f"  the ceiling Acc*:                                       {metrics['oracle_acc']:.4f}")
print(f"accuracy against the gold ordering, untied pairs:         {metrics['acc_gold_order_untied']:.4f}")
print(f"Spearman correlation of r_phi with g, held-out responses: {metrics['spearman_gold']:.4f}")

In [ ]:
# Checkpoint 4b. Thresholds set 2026-10-06 from seeds 0, 1 and 2 on the committed files (Apple M1
# Pro CPU; not yet checked on Colab or a GPU): the worst seed, with a margin of half the spread across
# seeds, rounded outward (briefs/09). Measured (seeds 0 / 1 / 2): Acc* - held-out accuracy 0.0237 /
# 0.0147 / 0.0417; gold-order accuracy 0.9381 / 0.9343 / 0.9250; Spearman 0.7848 / 0.7781 / 0.7454.
workshop.checkpoint(4, label="4b")
ACC_MARGIN = 0.06          # held-out accuracy >= Acc* - ACC_MARGIN; 0.0417 + 0.0270 / 2, rounded up
GOLD_ORDER_FLOOR = 0.91    # 0.9250 - 0.0131 / 2, rounded down
SPEARMAN_FLOOR = 0.72      # 0.7454 - 0.0394 / 2, rounded down
# Build-only: NLP_LLMS_LAB09_MEASURE=1 prints instead of asserting, so that the seed protocol can
# measure the metrics (and write each seed's checkpoint) before the thresholds are set.
MEASURE_ONLY = os.environ.get("NLP_LLMS_LAB09_MEASURE") == "1"
if OFFLINE_TINY:
    assert metrics["heldout_acc"] > 0.55, "the training loop should learn something even on the stand-in"
    print("Checkpoint 4b (stand-in mode): training ran; the thresholds are not applied to stand-in data")
elif MEASURE_ONLY:
    print("Checkpoint 4b NOT CHECKED (NLP_LLMS_LAB09_MEASURE=1):", {k: round(v, 4) for k, v in metrics.items()})
else:
    assert metrics["heldout_acc"] >= metrics["oracle_acc"] - ACC_MARGIN, metrics
    assert metrics["acc_gold_order_untied"] >= GOLD_ORDER_FLOOR, metrics
    assert metrics["spearman_gold"] >= SPEARMAN_FLOOR, metrics
    print(f"Checkpoint 4b passed: held-out accuracy {metrics['heldout_acc']:.3f} against the ceiling "
          f"{metrics['oracle_acc']:.3f}; the reward model ranks held-out responses much as g does")

**Explain.** Held-out pairwise accuracy is measured against the rater's noisy labels, so it sits near $\mathrm{Acc}^\star$, not near 1. Against the gold ordering of untied pairs, which is the question we actually care about, accuracy should be much higher: if so, the reward model learned "positive words up, negative words down" from comparisons alone, without ever seeing a score. That is the claim this checkpoint supports, and it is only a claim about responses like GPT-2's own samples, on frames it had not seen.

**Run: a preview of Module 10.** The provided cell scores five hand-written responses that GPT-2 rarely writes, after the first held-out prompt, and prints $r_\phi$ beside $g$. For each, it also prints the share of held-out reference responses that $r_\phi$ scores lower.

**Predict.** For "one positive word repeated 24 times", will $r_\phi$ be high or low? And $g$?

In [ ]:
# Provided: the preview. Printed only, never asserted: these responses are outside the data.
preview_prompt = prompts_doc["prompts"]["heldout"][0]
heldout_raw = score(reward_model, torch.cat([heldout["prompt_ids"]] * 2),
                    torch.cat([heldout["chosen_ids"], heldout["rejected_ids"]]))
print(f"Prompt: {preview_prompt['text']!r}\n")
print(f"{'response':<44} {'g':>6} {'r_phi':>7} {'above % of held-out':>20}")
preview_rows = []
for item in prompts_doc["preview"]:
    r_phi = score(reward_model, torch.tensor([preview_prompt["ids"]]), torch.tensor([item["response_ids"]])).item()
    gold = gold_reward(preview_prompt["text"], item["response"])
    above = (heldout_raw < r_phi).float().mean().item()
    preview_rows.append({"name": item["name"], "gold": gold, "r_phi_raw": r_phi, "share_of_heldout_below": above})
    print(f"{item['name']:<44} {gold:>6.2f} {r_phi:>7.2f} {above:>19.1%}")
    print(f"    {item['response'][:90]!r}")

**Explain.** Compare the two columns. The reward model learned from GPT-2's own samples, which almost never repeat a word or stack many list words, so the cap and the crowding penalty of $g$ were invisible in its training data. Where a hand-written response triggers them, $r_\phi$ and $g$ can disagree sharply: a response stuffed with positive words can score near the top of $r_\phi$ while $g$ calls it bad. Lab 10 optimizes a policy against $r_\phi$, and an optimizer finds exactly such responses. This is Goodhart's law in miniature (briefing section 8).

**Optional: score your own reply.** If the GPT-2 tokenizer can be downloaded from the Hugging Face Hub, the next cell scores a reply you write (at most 24 tokens are used). It is skipped otherwise.

In [ ]:
MY_REPLY = " happy, because it was the best day of my life."  # edit me

try:
    if OFFLINE_TINY:
        raise RuntimeError("offline stand-in mode")
    from transformers import AutoTokenizer

    gpt2_tokenizer = AutoTokenizer.from_pretrained(CAUSAL_LM, revision=prompts_doc["revision"])
    _ids = gpt2_tokenizer(MY_REPLY, add_special_tokens=False)["input_ids"][:RESPONSE_LEN]
    _text = gpt2_tokenizer.decode(_ids, clean_up_tokenization_spaces=False)
    _r_phi = score(reward_model, torch.tensor([preview_prompt["ids"]]), torch.tensor([_ids])).item()
    print(f"{preview_prompt['text']!r} + {_text!r}")
    print(f"g = {gold_reward(preview_prompt['text'], _text):.2f} · r_phi = {_r_phi:.2f} · above "
          f"{(heldout_raw < _r_phi).float().mean().item():.1%} of held-out responses")
except Exception as error:  # no network, no transformers, or the stand-in: skip
    print(f"Skipped: the GPT-2 tokenizer did not load ({type(error).__name__}); this preview needs the Hub.")
TIMES["Exercise 4 evaluation"] = time.time() - LAB_START - sum(TIMES.values())

## Exercise 5 · Normalize and Save the Reward Model (6 minutes)

Only differences of $r_\phi$ are identified, so its level is arbitrary. Before Lab 10 uses it as a reward, fix the level and the scale on a reference set: the held-out responses, both responses of every held-out pair (briefing eq. `normalize`):

$$
\tilde r_\phi(x, y) = \frac{r_\phi(x, y) - \mu}{\varsigma},
$$

with $\mu$ and $\varsigma$ the mean and the **population** standard deviation (divide by $n$, not $n - 1$) of the raw scores. Lab 10's reward-gain thresholds are in these units: "+1" means one standard deviation of the reference policy's own responses.

**Predict.** Does normalizing change any pairwise accuracy from Exercise 4?

**Task.** Complete `fit_normalizer(raw_scores)`: a 1-D tensor in, `(mean, std)` out, as Python floats.

In [ ]:
# TODO 5: complete this function.
def fit_normalizer(raw_scores):
    """(mean, population std) of a 1-D tensor of raw reward scores, as Python floats."""
    raise NotImplementedError("TODO 5")

In [ ]:
#@title Solution 5 — try it yourself first { display-mode: "form" }
@workshop.solution(5)
def fit_normalizer(raw_scores):
    """(mean, population std) of a 1-D tensor of raw reward scores, as Python floats."""
    raw_scores = raw_scores.double()
    return raw_scores.mean().item(), raw_scores.std(unbiased=False).item()  # divide by n

<details>
<summary>Why this solution works</summary>

Subtracting $\mu$ and dividing by $\varsigma > 0$ is an increasing affine map, so every comparison, every accuracy and the Spearman correlation are unchanged. What changes is the reward's level and unit, which matter downstream: the policy gradient uses the reward's level unless a baseline removes it, and Lab 10's KL coefficient only makes sense against a reward of known scale. `unbiased=False` gives the population standard deviation, so the normalized reference scores have standard deviation exactly 1. Computing in float64 keeps the check below at its tolerance.

</details>

In [ ]:
# Checkpoint 5: normalization, then save, reload and compare.
workshop.checkpoint(5)
_mean, _std = fit_normalizer(torch.tensor([1.0, 2.0, 3.0, 6.0]))
assert abs(_mean - 3.0) < 1e-9 and abs(_std - math.sqrt(3.5)) < 1e-9, (
    f"the mean and population std of [1, 2, 3, 6] are 3.0 and sqrt(3.5) = {math.sqrt(3.5):.4f}; "
    f"got {_mean} and {_std}: divide by n, not n - 1")

score_mean, score_std = fit_normalizer(heldout_raw)
reward_model.score_mean.fill_(score_mean)
reward_model.score_std.fill_(score_std)
_normalized = score(reward_model, torch.cat([heldout["prompt_ids"]] * 2),
                    torch.cat([heldout["chosen_ids"], heldout["rejected_ids"]]), normalized=True)
assert abs(_normalized.double().mean().item()) < 1e-4 and abs(_normalized.double().std(unbiased=False).item() - 1) < 1e-4
assert evaluate_reward_model(reward_model, heldout) == metrics, "normalizing changes no comparison"

checkpoint = {
    "format": "nlp-llms/lab09-reward-model", "version": 2,
    "config": {"n_rows": n_rows, **RM_CONFIG},
    "policy": {"model": prompts_doc["model"], "revision": prompts_doc["revision"], "eos_id": EOS_ID,
               "vocab_size": EOS_ID + 1, "prompt_len": PROMPT_LEN, "response_len": RESPONSE_LEN},
    "state_dict": reward_model.state_dict(),  # includes vocab_map, score_mean and score_std
    "gold": copy.deepcopy(GOLD),
    "metrics": {k: round(v, 4) for k, v in metrics.items()},
    "data_sha256": data_sha256,
}
# The repository's copy, data/lab09_reward_model.pt, is written by this cell when the build sets
# NLP_LLMS_REWARD_MODEL_OUT; everyone else writes to the working directory.
CHECKPOINT_PATH = Path(os.environ.get("NLP_LLMS_REWARD_MODEL_OUT", "lab09_reward_model.pt"))
STAND_IN_DATA = OFFLINE_TINY or prompts_doc.get("build", {}).get("stand_in", False)
if STAND_IN_DATA and (OFFLINE_TINY and "NLP_LLMS_REWARD_MODEL_OUT" in os.environ
                      or CHECKPOINT_PATH.resolve().parent.name == "data"):
    raise RuntimeError("refusing to write a reward model trained on stand-in data to that path")
torch.save(checkpoint, CHECKPOINT_PATH)

loaded = torch.load(CHECKPOINT_PATH, weights_only=True)
fresh = RewardModel(**loaded["config"]).eval()
fresh.load_state_dict(loaded["state_dict"])
_b = slice(0, 64)
with torch.no_grad():
    _out_saved = reward_model(heldout["prompt_ids"][_b], heldout["chosen_ids"][_b])
    _out_fresh = fresh(heldout["prompt_ids"][_b], heldout["chosen_ids"][_b])
assert _out_fresh.shape == (64,) and _out_fresh.dtype == torch.float32
assert torch.equal(_out_fresh, _out_saved), "the reloaded model must give identical outputs"
assert loaded["gold"] == GOLD and loaded["policy"]["eos_id"] == EOS_ID
size_mb = CHECKPOINT_PATH.stat().st_size / 1e6
print(f"Checkpoint 5 passed: normalized held-out scores have mean 0 and std 1; saved {CHECKPOINT_PATH} "
      f"({size_mb:.2f} MB), reloaded with identical outputs")
print(f"score_mean = {score_mean:.4f}, score_std = {score_std:.4f}")
TIMES["Exercise 5"] = time.time() - LAB_START - sum(TIMES.values())

**Explain.** The file holds everything Lab 10 needs to rebuild the reward model exactly: the configuration, the weights, the compact vocabulary, the normalization, the gold rule it was checked against and the hashes of the data it was trained on. Lab 10 freezes it (`eval()`, no gradients): the policy gradient treats its score as a number, as Part A treated the toy reward.

**If you trained your own.** Lab 10 loads the workshop's copy of this file by default. To use yours, download `lab09_reward_model.pt` from the Files panel and upload it to Lab 10's runtime.

In [ ]:
# The results card: every number measured in this run.
card = {
    "offline_stand_in": OFFLINE_TINY,
    "device": DEVICE if DEVICE == "cpu" else torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "part_a": {
        "y_star": y_star.tolist(), "K": K, "N": N,
        "cosine_mean_vs_exact": {k: round(cosine(v.mean(0), g_flat), 4) for k, v in estimates.items()},
        "total_variance": {k: round(v, 5) for k, v in total_variance.items()},
        "final_J_no_baseline_seed0": round(history_none[-1], 4),
        "steps_to_0.9": {name: [first for _, first in results] for name, results in runs.items()},
        "final_J": {name: [round(h[-1], 4) for h, _ in results] for name, results in runs.items()},
    },
    "part_b": {
        "data_model": prompts_doc["model"], "data_revision": prompts_doc["revision"],
        "data_sha256": data_sha256, "rm_seed": RM_SEED, "rm_train": RM_TRAIN, "best_epoch": best_epoch,
        "rm_history": rm_history, "n_rows": n_rows, "tie_share_heldout": round(tie_share, 4),
        "metrics": checkpoint["metrics"], "score_mean": score_mean, "score_std": score_std,
        "checkpoint_mb": round(size_mb, 3), "preview": preview_rows,
    },
    "seconds": {k: round(v, 1) for k, v in TIMES.items()},
}
with open("lab09_results.json", "w", encoding="utf-8") as f:
    json.dump(card, f, indent=1)
print("Run time by section (s):", {k: round(v) for k, v in TIMES.items()})
print(f"\nThis is the end of the core path. Run time so far: {time.time() - LAB_START:.0f} s")
if OFFLINE_TINY:
    print("OFFLINE STAND-IN MODE: Part B numbers are not results.")

## Stretch (optional) · Noisier Raters

For those who finish early. Lab 10 does not depend on this section.

Section 7 of the briefing said that scale and noise are tangled: from comparisons we learn $u / \tau_{\text{label}}$, so noisier raters should give a reward model with smaller differences but, given enough data, the same ranking. With finite data, noise also costs information. The cell below relabels the **training** pairs from their stored gold scores at $\tau_{\text{label}} \in \{0.25, 0.5, 1, 2, 4\}$ (a fixed seed), retrains a reward model with a shorter budget for each, and measures accuracy against the gold ordering on the held-out pairs. The held-out pairs keep the file's labels.

**Predict.** Sketch the curve of gold-order accuracy against $\tau_{\text{label}}$ before you run it. Where will it fall fastest?

In [ ]:
def relabel(data, tau_label, seed=SEED):
    """A copy of the pairs, chosen/rejected redrawn from bt_prob(g_chosen, g_rejected, tau_label)."""
    generator = torch.Generator().manual_seed(seed)
    keep = torch.rand(len(data["gold_chosen"]), generator=generator) < bt_prob(
        data["gold_chosen"], data["gold_rejected"], tau_label)  # True: the stored order stands
    out = dict(data)
    for a, b in (("chosen_ids", "rejected_ids"), ("gold_chosen", "gold_rejected")):
        k = keep[:, None] if data[a].dim() == 2 else keep
        out[a], out[b] = torch.where(k, data[a], data[b]), torch.where(k, data[b], data[a])
    return out


STRETCH_TAUS = [0.25, 0.5, 1.0, 2.0, 4.0]
STRETCH_EPOCHS = 3
started = time.time()
stretch = []
for tau in STRETCH_TAUS:
    model_tau, _, _ = train_reward_model(relabel(train, tau), vocab_map, n_rows, epochs=STRETCH_EPOCHS,
                                         val_pairs=0, verbose=False)
    m = evaluate_reward_model(model_tau, heldout)
    stretch.append({"tau_label": tau, "acc_gold_order_untied": m["acc_gold_order_untied"],
                    "spearman_gold": m["spearman_gold"],
                    "oracle_acc_at_tau": oracle_accuracy(h_c, h_r, tau)})
    print(f"tau_label {tau:<5} gold-order accuracy {m['acc_gold_order_untied']:.4f} · Spearman "
          f"{m['spearman_gold']:.4f} · Acc* at this tau {stretch[-1]['oracle_acc_at_tau']:.4f}")
print(f"({time.time() - started:.0f} s)")

plt.figure(figsize=(6, 3.4))
plt.plot(STRETCH_TAUS, [s["acc_gold_order_untied"] for s in stretch], "o-", color=ACCENT,
         label="reward model: accuracy against the gold ordering")
plt.plot(STRETCH_TAUS, [s["oracle_acc_at_tau"] for s in stretch], "s--", color=GRAY,
         label="Acc* of the rater at this temperature")
plt.xscale("log")
plt.xticks(STRETCH_TAUS, [str(t) for t in STRETCH_TAUS])
plt.xlabel("label temperature tau_label (training labels)")
plt.ylabel("held-out")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()
assert len(stretch) == len(STRETCH_TAUS)

**Explain.** As the raters get noisier, more training labels point the wrong way, and the reward model recovers the gold ordering less well; the rater's own ceiling $\mathrm{Acc}^\star$ falls toward 0.5 at the same time. The reward model's gold-order accuracy can stay well above $\mathrm{Acc}^\star$, because it averages over many noisy labels, but with a fixed number of pairs the information runs out. More data or less noisy raters: in practice, this is the cost trade-off of collecting preferences.

**Optional: flipped labels.** Instead of a higher temperature, flip a share $\varepsilon$ of the file's training labels at random. Nothing is asserted.

In [ ]:
def flip_labels(data, epsilon, seed=SEED):
    generator = torch.Generator().manual_seed(seed + 1)
    flip = torch.rand(len(data["gold_chosen"]), generator=generator) < epsilon
    out = dict(data)
    for a, b in (("chosen_ids", "rejected_ids"), ("gold_chosen", "gold_rejected")):
        k = flip[:, None] if data[a].dim() == 2 else flip
        out[a], out[b] = torch.where(k, data[b], data[a]), torch.where(k, data[a], data[b])
    return out


for epsilon in (0.1, 0.3):
    model_eps, _, _ = train_reward_model(flip_labels(train, epsilon), vocab_map, n_rows,
                                         epochs=STRETCH_EPOCHS, val_pairs=0, verbose=False)
    m = evaluate_reward_model(model_eps, heldout)
    print(f"epsilon {epsilon}: gold-order accuracy {m['acc_gold_order_untied']:.4f} · Spearman {m['spearman_gold']:.4f}")

In [ ]:
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Which checkpoints passed, and on whose code; then a run record for the instructor.
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
---
**Next:** [10 · RLHF](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/10-rlhf.ipynb)

[Workshop site](https://project-delphi.github.io/nlp-llms) · [All notebooks](https://project-delphi.github.io/nlp-llms/notebooks.html) · [Source](https://github.com/project-delphi/nlp-llms)

Genial Labs · text CC BY 4.0, code MIT